# Introduction to Polars

Companion notebook for the *Introduction to Polars* session: every example of the slides, and
more, on real power-system data. It is written Polars first. pandas appears only where the
difference matters (the index, date guessing, missing values, alignment, speed), and section 5
is a map for those who come from pandas.

| Part | Content |
|------|---------|
| 0 | Setup: environment check, download of the dataset, the three tables used everywhere |
| 1 | **Polars in ten minutes**: data model, contexts, expressions |
| 2 | **Basics**: inspecting, dates, cleaning, basic operations, combining, grouping and reshaping |
| 3 | **Time series**: resampling and frequency, window functions, changes and lags |
| 4 | **The lazy engine and the rest**: lazy API, Polars 2.0 streaming engine, speed measured against pandas, pitfalls, interop, PySpark |
| 5 | Coming from pandas: a cheat sheet |
| 6 | Three longer exercises |

**Dataset.** Open Power System Data, *Time series* package, version 2020-10-06, 30-minute
resolution: electricity load, wind and solar generation for Great Britain, Northern Ireland,
Ireland and Cyprus, 2015 to September 2020 (100,802 rows, 41 columns, 37 MB CSV).

> Open Power System Data. 2020. Data Package Time series. Version 2020-10-06.
> https://doi.org/10.25832/time_series/2020-10-06. (Primary data from various sources, for a
> complete list see URL.)

**How to use it in the session.** Open it and choose *Runtime > Run all* (*Run > Run All
Cells* in JupyterLab) while the introduction is on screen: every table and variable is then
ready, and you can change any cell and run it again. Section 1.2 and each section of parts 2 and 3
end with a short **Try it** task (three to five minutes) and an empty cell to work in. The solution is
folded under the task: nothing is revealed until you click *Show the solution*.

**Environment.** The notebook needs **Polars 2.0 or newer** and runs with pandas 2.2 or 3.0, on
Python 3.12 or 3.13.

- **Google Colab**: nothing to prepare. Colab ships an older Polars; the first code cell detects
  that and upgrades it, and the dataset is downloaded by the notebook. Use *Runtime > Run all*.
- **Locally**, with `uv`:

```
uv add "polars>=2.0" pandas pyarrow numpy matplotlib plotly jupyterlab
```

## 0. Setup

Run this cell first. It checks the installed Polars version without importing it. On Colab it
upgrades Polars when needed; anywhere else it stops with the command to run, because a notebook
should not silently modify a local environment.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys

MIN_POLARS = (2, 0)
POLARS_REQUIREMENT = "polars>=2.0,<3"


def installed_version(package: str) -> tuple[int, ...] | None:
    """Return the installed (major, minor) version of a package, or None if it is missing."""
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None
    return tuple(int(part) for part in version.split(".")[:2])


def running_in_colab() -> bool:
    """Detect a Google Colab runtime."""
    return "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ


def ensure_polars() -> None:
    """Make sure Polars >= 2.0 is installed; upgrade it automatically on Colab only."""
    current = installed_version("polars")
    if current is not None and current >= MIN_POLARS:
        print(f"Polars {importlib.metadata.version('polars')} found: nothing to install.")
        return

    found = "not installed" if current is None else importlib.metadata.version("polars")
    if not running_in_colab():
        raise RuntimeError(
            f"This notebook needs Polars >= 2.0 (found: {found}). "
            f'Install it with: uv add "{POLARS_REQUIREMENT}"  (or pip install "{POLARS_REQUIREMENT}")'
        )

    already_imported = "polars" in sys.modules
    print(f"Colab runtime with Polars {found}: upgrading to {POLARS_REQUIREMENT} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", POLARS_REQUIREMENT])
    importlib.invalidate_caches()
    print(f"Polars {importlib.metadata.version('polars')} installed.")
    if already_imported:
        raise RuntimeError(
            "Polars was upgraded, but the old version is already loaded in this session. "
            "Use Runtime > Restart session, then Runtime > Run all."
        )


ensure_polars()

In [ ]:
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.request import urlretrieve
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
import polars.selectors as cs

pl.Config.set_tbl_cols(12)   # show up to 12 columns before eliding with "..."
UTC = timezone.utc

print(
    f"python {sys.version.split()[0]} | polars {pl.__version__} | "
    f"pandas {pd.__version__} | numpy {np.__version__}"
)
print(f"threads available to Polars: {pl.thread_pool_size()}")

### Download the dataset

The file is public and is downloaded once into `data/`. If the download is not possible from
your network, download the CSV by hand from the URL below and put it in the `data/` folder.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

OPSD_URL = (
    "https://data.open-power-system-data.org/time_series/2020-10-06/"
    "time_series_30min_singleindex.csv"
)
CSV_PATH = DATA_DIR / "time_series_30min_singleindex.csv"


def download_dataset(url: str, destination: Path) -> None:
    """Download the dataset once; keep partial downloads out of the way."""
    if destination.exists():
        print(f"{destination} already present ({destination.stat().st_size / 1e6:.1f} MB)")
        return
    partial = destination.with_suffix(".part")
    try:
        urlretrieve(url, partial)
    except OSError as err:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f"Could not download the dataset ({err}). Download it manually from {url} "
            f"and save it as {destination}."
        ) from err
    partial.replace(destination)
    print(f"downloaded {destination} ({destination.stat().st_size / 1e6:.1f} MB)")


download_dataset(OPSD_URL, CSV_PATH)

### The three tables used in the notebook

How the file is read, and why the schema is declared explicitly, is explained in section 2.1.
For now we only need the tables:

- `power`: the file as it is, wide, one column per series (41 columns);
- `gb`: Great Britain only, five tidy columns;
- `regional`: load and onshore wind for four regions, in long format (one row per region and
  timestamp), plus `regions`, a small metadata table.

In [ ]:
header = pl.read_csv(CSV_PATH, n_rows=0).columns
schema_overrides = {name: pl.Float64 for name in header if not name.endswith("timestamp")}

power = pl.read_csv(CSV_PATH, schema_overrides=schema_overrides, try_parse_dates=True)

gb = power.select(
    timestamp="utc_timestamp",
    load_mw="GB_GBN_load_actual_entsoe_transparency",
    load_forecast_mw="GB_GBN_load_forecast_entsoe_transparency",
    wind_mw="GB_GBN_wind_generation_actual",
    solar_mw="GB_GBN_solar_generation_actual",
)

REGION_CODES = ["GB_GBN", "GB_NIR", "IE", "CY"]
regional = pl.concat(
    [
        power.select(
            timestamp="utc_timestamp",
            region=pl.lit(code),
            load_mw=f"{code}_load_actual_entsoe_transparency",
            wind_onshore_mw=f"{code}_wind_onshore_generation_actual",
        )
        for code in REGION_CODES
    ]
)

regions = pl.DataFrame(
    {
        "region": REGION_CODES,
        "name": ["Great Britain", "Northern Ireland", "Ireland", "Cyprus"],
        "country": ["United Kingdom", "United Kingdom", "Ireland", "Cyprus"],
    }
)

print(f"power: {power.shape} | gb: {gb.shape} | regional: {regional.shape} | regions: {regions.shape}")
gb.slice(60, 3)

# Part 1 - Polars in ten minutes

Three ideas are enough to read any Polars code: the data model, the contexts, and expressions.

## 1.1 The data model: typed columns, no index

A `DataFrame` is a collection of typed columns (`Series`). Compared with pandas:

- every column has **one strict data type**, shown in the header;
- there is **no index** and there are no row labels: a row is identified by its position, and
  keys such as the timestamp are ordinary columns;
- operations **return a new frame**; nothing is modified in place.

In [ ]:
print(gb.shape)
print(gb.schema)

gb.slice(60, 3)

In [ ]:
# Square brackets exist, but only for quick inspection. A single column is a Series.
load = gb["load_mw"]
print(type(load).__name__, load.dtype, load.len())

print(gb.row(60, named=True))     # one row as a dict

### What "no index" means

In pandas every row carries a **label**, and the labels live apart from the columns, in the
*index*: `0, 1, 2, ...` by default, the timestamps after `set_index("timestamp")`. Selection by
label (`.loc`), `resample`, `rolling` and the alignment of arithmetic all go through it.
The same three rows, as pandas holds them once the timestamp is the index:

In [ ]:
sample_pd = gb.slice(60, 3).select("timestamp", "load_mw", "wind_mw").to_pandas().set_index("timestamp")

print(sample_pd)
print()
print("index  :", type(sample_pd.index).__name__)
print("columns:", list(sample_pd.columns), "<- the timestamp is no longer one of them")
print("by label:", sample_pd.loc["2015-01-02 05:30", "load_mw"])

Polars has no such thing. The timestamp stays an ordinary column and a row has only a position,
so the same lookup is a `filter` on that column. Wherever pandas uses the index, Polars asks you
to **name the column**: `sort("timestamp")`, `join(on="timestamp")`,
`group_by_dynamic("timestamp", ...)`, `rolling_mean_by("timestamp", ...)`.

In [ ]:
gb.filter(pl.col("timestamp") == datetime(2015, 1, 2, 5, 30, tzinfo=UTC)).select("timestamp", "load_mw")

## 1.2 The contexts

Almost everything in Polars is written as *context(expressions)*. A context decides **where**
an expression is evaluated; the expression says **what** to compute.

| Context | What it does |
|---------|--------------|
| `select` | choose / compute columns, drop the rest |
| `with_columns` | add or replace columns, keep the rest |
| `filter` | keep rows where a boolean expression is true |
| `group_by(...).agg(...)` | one row per group |
| `sort` | order rows |

A good habit: **say the query in words first**, then write one context per step. The question
"the mean load and the mean wind plus solar output, for each month of 2019" becomes:

1. keep the rows of 2019;
2. add a column: wind plus solar;
3. make one group per calendar month;
4. take the means of each group;
5. put the months in order.

In [ ]:
(
    gb
    .filter(pl.col("timestamp").dt.year() == 2019)                           # 1
    .with_columns(renewables_mw=pl.col("wind_mw") + pl.col("solar_mw"))      # 2
    .group_by(month=pl.col("timestamp").dt.month())                          # 3
    .agg(                                                                    # 4
        load_mean=pl.col("load_mw").mean(),
        renewables_mean=pl.col("renewables_mw").mean(),
    )
    .sort("month")                                                           # 5
    .head(4)
)

The parentheses around the chain are the usual Python style for Polars: one operation per line,
readable top to bottom. Note the explicit `sort` at the end: **`group_by` does not return the
groups in a defined order**.

### Try it 1.2: your first query, from words to code

The question: **for each hour of the day, the mean and the highest load of 2019 in Great
Britain**.

1. Split the question into steps, in words, as above: which rows, which groups, which numbers,
   which order.
2. Write one context per step, starting from `gb`. Hint: `pl.col("timestamp").dt.hour()` gives
   the hour of a timestamp.
3. At what hour is the mean load highest, and at what hour is it lowest?

**Expected result**

```raw
shape: (24, 3)
┌──────┬──────────────┬──────────┐
│ hour ┆ load_mean    ┆ load_max │
│ ---  ┆ ---          ┆ ---      │
│ i8   ┆ f64          ┆ f64      │
╞══════╪══════════════╪══════════╡
│ 0    ┆ 26374.314384 ┆ 35291.0  │
│ 1    ┆ 25673.55     ┆ 34528.0  │
│ 2    ┆ 24656.568082 ┆ 32904.0  │
│ 3    ┆ 24075.703379 ┆ 32419.0  │
│ 4    ┆ 24352.133333 ┆ 31906.0  │
│ …    ┆ …            ┆ …        │
│ 19   ┆ 38877.841096 ┆ 50954.0  │
│ 20   ┆ 36855.572603 ┆ 49397.0  │
│ 21   ┆ 33908.230822 ┆ 45651.0  │
│ 22   ┆ 30325.679452 ┆ 40433.0  │
│ 23   ┆ 27331.766438 ┆ 36026.0  │
└──────┴──────────────┴──────────┘
```


<details>
<summary><b>Show the solution</b></summary>

```python
(
    gb
    .filter(pl.col("timestamp").dt.year() == 2019)             # 1 keep the rows of 2019
    .group_by(hour=pl.col("timestamp").dt.hour())              # 2 one group per hour of the day
    .agg(                                                      # 3 two numbers per group
        load_mean=pl.col("load_mw").mean(),
        load_max=pl.col("load_mw").max(),
    )
    .sort("hour")                                              # 4 hours in order
)
```

- Four steps in words, four contexts in code. Everything written *inside* `filter`, `group_by`
  and `agg` is an expression: that is the subject of the next section.
- The mean load is lowest in the 03:00 hour (24.1 GW) and highest in the 17:00 hour (40.3 GW),
  the evening peak. The highest single value of 2019, 51,467 MW, falls in the 09:00 hour.
- Without the final `sort` the 24 rows come back in no defined order, and "the first row" means
  nothing.
- The timestamps are in UTC, so these are UTC hours: British clocks are one hour ahead in
  summer. Local time is handled in section 2.2.



In [ ]:
# Your code here

## 1.3 Expressions

An **expression** (`pl.Expr`) is a *lazy representation of a data transformation*: a description
of a computation on columns. It holds no data, and it becomes a result only when a context runs
it. Expressions are also **building blocks**: small ones combine into bigger ones.

In [ ]:
load_gw = pl.col("load_mw") / 1000

print(type(load_gw))
print(load_gw)                      # the expression tree, no data involved
print(load_gw.meta.root_names())    # which columns it needs

In [ ]:
# The same recipe, run in three different contexts.
print(gb.select(load_gw.alias("load_gw")).slice(60, 2))
print(gb.filter(load_gw > 55).height, "half-hours above 55 GW")
print(
    gb.group_by(year=pl.col("timestamp").dt.year())
    .agg(load_gw.max().alias("peak_gw"))
    .sort("year")
    .tail(3)
)

The same holds for any piece of logic: write it once, give it a name, use it in every context.
Here, what "a missing load value" means, on the long `regional` table. This is the second reason
to use Polars, next to speed: one consistent way to write things.

In [ ]:
missing = pl.col("load_mw").is_null()                                    # one recipe

print(regional.filter(missing).height, "rows with a missing load, of", regional.height)   # which rows
print(regional.select(missing.sum()))                                    # how many
print(regional.group_by("region").agg(missing.sum()).sort("region"))     # per region

In [ ]:
# Building blocks: two small recipes combined into a third. Still no data involved.
renewables_gw = (pl.col("wind_mw") + pl.col("solar_mw")) / 1000
share = renewables_gw / load_gw

print(share)
print(share.meta.root_names())

# Only a context computes something: the median share of wind plus solar in the load.
gb.select(share.median())

The result column is called `wind_mw`: an expression without a name keeps the name of its first
root column. Naming the output is the third part of every expression.

The anatomy of an expression is always the same:

```
pl.col("load_mw")   .fill_null(strategy="forward")   .rolling_mean(48)   .alias("load_daily")
\__ start __/        \_________ chain of operations _________/           \_ output name _/
```

- **start** from one or more columns (`pl.col`), a literal (`pl.lit`) or a selector;
- **chain** operations, each returning a new expression;
- **name** the output (`.alias`, or a keyword argument in the context). Without a name, the
  output keeps the name of the root column, which is how `with_columns` *replaces* a column.

### The same recipe, eager or lazy

Everything so far ran on a `DataFrame`: each call computed at once, on data held in memory. The
same expressions and the same contexts also work on a `LazyFrame`, which only records what you
ask for and runs it when you call `collect()`. Switching takes two edits: `scan_parquet` instead
of `read_parquet`, and `collect()` at the end.

In [ ]:
GB_PATH = DATA_DIR / "gb.parquet"
gb.write_parquet(GB_PATH)                      # the tidy Great Britain table, as a Parquet file

df = pl.read_parquet(GB_PATH)                  # eager: the data is in memory
lf = pl.scan_parquet(GB_PATH)                  # lazy: a plan, nothing has been read yet
print(type(df).__name__, "|", type(lf).__name__, "|", type(share).__name__)

print(df.select(share.median()))               # runs now
print(lf.select(share.median()).collect())     # runs at collect()

In [ ]:
# The plan of the lazy version: it reads only the three columns the recipe needs.
print(lf.select(share.median()).explain())

The classes behind all of this, plus the one you get when you pick a single column:

| Class | What it is | How you get one |
|-------|------------|-----------------|
| `pl.DataFrame` | a table in memory; each method runs at once | `pl.read_*`, `pl.DataFrame({...})`, `lf.collect()` |
| `pl.LazyFrame` | a query plan; nothing runs until `collect()` | `pl.scan_*`, `df.lazy()` |
| `pl.Expr` | a recipe on columns; the same object works in both | `pl.col(...)`, `pl.lit(...)`, selectors |
| `pl.Series` | one typed column | `df["load_mw"]`, `pl.Series([...])` |

Part 4 comes back to the lazy engine: what the optimiser does, and how much it gains.

### Composing and reusing expressions

Expressions are ordinary Python objects: store them in variables, combine them, and write
functions that return them.

In [ ]:
def zscore(column: str) -> pl.Expr:
    """Standardise a column: (x - mean) / std."""
    col = pl.col(column)
    return (col - col.mean()) / col.std()


# The four regions have very different sizes: a z-score over the whole table mostly measures
# "which region is this". Per region, with .over(), it measures "is this a high-load moment".
(
    regional
    .with_columns(
        z_all=zscore("load_mw").round(3),
        z_region=zscore("load_mw").over("region").round(3),
    )
    .drop("wind_onshore_mw")
    .filter(pl.col("timestamp") == datetime(2019, 1, 15, 18, 0, tzinfo=UTC))
)

### Expressions written by code

Because an expression is an object, ordinary Python can *produce* expressions: a comprehension
over a dictionary of windows gives a list of rolling means, each with its own name. Nothing is
computed while the list is built.

In [ ]:
WINDOWS = {"3h": 6, "1d": 48, "7d": 336}      # label -> number of half-hourly samples

rolling = [
    pl.col("load_mw").rolling_mean(n).alias(f"load_mean_{name}") for name, n in WINDOWS.items()
]

print(len(rolling), "objects of type", type(rolling[0]).__name__)
for expr in rolling:
    print(f"  {expr.meta.output_name():<14} <- {expr}")

The same idea turns a **table of limits** into one boolean expression, in two steps. `checks`
is a list of three expressions, one true/false column per signal. `pl.all_horizontal` then
combines them **across the columns, row by row**, with a logical AND: the result is true on a
row only if every check is true on that row (`any_horizontal` is the OR; the ordinary `all()`
works down a single column). Writing `checks[0] & checks[1] & checks[2]` gives the same
result; `all_horizontal` takes a list of any length.

In [ ]:
LIMITS = {"load_mw": (15_000, 60_000), "wind_mw": (0, 20_000), "solar_mw": (0, 15_000)}

checks = [pl.col(c).is_between(lo, hi) for c, (lo, hi) in LIMITS.items()]   # one true/false per signal
in_range = pl.all_horizontal(checks)                                        # true only if all are true

# What "horizontal" means, on four rows: across the columns of each row. Nulls stay unknown.
print(
    pl.DataFrame({"a": [True, True, False, None], "b": [True, False, None, True]}).with_columns(
        both=pl.all_horizontal("a", "b"), either=pl.any_horizontal("a", "b")
    )
)
print("reads the columns:", in_range.meta.root_names())

# One context runs everything: three generated columns, the z-score function and the flag.
engineered = gb.with_columns(*rolling, z=zscore("load_mw"), ok=in_range)

print(engineered["ok"].value_counts(sort=True))
engineered.filter(pl.col("timestamp") == datetime(2019, 1, 15, 18, 0, tzinfo=UTC)).with_columns(cs.float().round(2))

Five new columns from three small definitions. The flag is `false` on 230 rows (implausible load
values: we meet them again in sections 2.4 and 3.3) and `null` on 124, where an input is missing.

Why this matters in practice: the windows and the limits could come from a configuration file
or from a table of signals. Adding a feature or a check is then a change of *data*, not of code,
and the same list of expressions works unchanged on a `LazyFrame`.

All the expressions passed to a single context are independent of each other and are
**executed in parallel**. That is why one `with_columns` with five expressions is preferred over
five chained calls, and why a column created in a context cannot be used *in the same* context:

In [ ]:
try:
    gb.with_columns(
        renewables_mw=pl.col("wind_mw") + pl.col("solar_mw"),
        renewable_share=pl.col("renewables_mw") / pl.col("load_mw"),   # does not exist yet
    )
except pl.exceptions.ColumnNotFoundError as err:
    print("ColumnNotFoundError:", str(err).splitlines()[0])

# Fix: a second context.
(
    gb
    .with_columns(renewables_mw=pl.col("wind_mw") + pl.col("solar_mw"))
    .with_columns(renewable_share=pl.col("renewables_mw") / pl.col("load_mw"))
    .select("timestamp", "load_mw", "renewables_mw", "renewable_share")
    .slice(60_000, 2)
)

### Many columns at once: selectors

An expression can start from several columns. `pl.col("a", "b")` takes names; **selectors**
(`polars.selectors`, imported as `cs`) pick columns by type or by name pattern and support set
operations (`|` union, `&` intersection, `-` difference, `~` complement).
`.name.suffix()` / `.name.prefix()` keep the outputs from colliding.

In [ ]:
print(gb.select(cs.numeric().mean().name.suffix("_mean")))

# 41 columns: selectors make the wide table manageable.
print(power.select(cs.contains("wind") & cs.ends_with("generation_actual")).columns)
print(power.select(cs.starts_with("IE_") - cs.contains("sem")).columns)

### When the name cannot come from the selector

A selector expands one expression into many, and each output takes its name from the **root
column**: the column the expression starts from. Ask "for every float column, the first
timestamp with a valid value" and the root is always the timestamp column, whatever the selector
inside `filter` expands to:

In [ ]:
ts_col = "utc_timestamp"

try:
    power.select(
        pl.col(ts_col).filter(cs.float().is_not_null()).min().name.suffix("_first_valid")
    )
except pl.exceptions.DuplicateError as err:
    print("DuplicateError:", str(err).splitlines()[0])

The selector did expand, to 39 expressions, and all 39 are called `utc_timestamp_first_valid`.
When the names have to come from somewhere else, let Python write the expressions:
`cs.expand_selector` returns the column names a selector matches, and a function returns the
recipes for one column.

In [ ]:
def valid_span(column: str) -> list[pl.Expr]:
    """First and last timestamp at which `column` is not null."""
    ts_valid = pl.col(ts_col).filter(pl.col(column).is_not_null())
    return [
        ts_valid.min().alias(f"{column}_first_valid"),
        ts_valid.max().alias(f"{column}_last_valid"),
    ]


float_cols = cs.expand_selector(power, cs.float())      # names of the Float32/Float64 columns
valid_wide = power.select(expr for column in float_cols for expr in valid_span(column))

print(valid_wide.shape)
valid_wide.select(cs.starts_with("CY_load_actual"))

One row and 78 columns: correct, but hard to read. A **long** table, one row per column, is
easier, and it can be built from the same kind of wide result. Let each column produce a single
**struct** that holds all its results; `unpivot` then turns the 39 struct columns into 39 rows,
and `unnest` turns the fields of the struct into ordinary columns.

In [ ]:
def valid_summary(column: str) -> pl.Expr:
    """First and last valid timestamp and number of valid samples of `column`, as one struct."""
    ts_valid = pl.col(ts_col).filter(pl.col(column).is_not_null())
    return pl.struct(
        first_valid=ts_valid.min(),
        last_valid=ts_valid.max(),
        n_valid=pl.col(column).count(),
    ).alias(column)


summary_wide = power.select(valid_summary(column) for column in float_cols)
print(summary_wide.shape, "| dtype of one column:", summary_wide.dtypes[0])

coverage = (
    summary_wide
    .unpivot(variable_name="column")      # 39 rows: the column name and its struct
    .unnest("value")                      # the struct fields become columns
)
coverage.sort("first_valid", descending=True).head(5)

Only 39 values are reshaped here, not the table: the aggregation ran first, in one parallel pass
over `power`. Section 2.1 builds the same table the other way round (reshape first, then
aggregate) and compares the two.

### Conditionals: `when / then / otherwise`

The vectorised if-else. A bare string inside `then()` / `otherwise()` is read as a **column
name**: wrap text in `pl.lit()`. Without `otherwise()`, rows matching no branch become `null`.

In [ ]:
level = (
    pl.when(pl.col("load_mw") > 45_000).then(pl.lit("high"))
    .when(pl.col("load_mw") < 25_000).then(pl.lit("low"))
    .when(pl.col("load_mw").is_null()).then(pl.lit("no data"))
    .otherwise(pl.lit("normal"))
)

gb.with_columns(level=level)["level"].value_counts(sort=True)

# Part 2 - Basics

Each section starts with a table: the task on the left, the Polars tool in the middle, a note on
the right. Then the tools are used on the real data.

## 2.1 Inspecting data

| Task | Polars | Note |
|------|--------|------|
| First and last rows | `df.head()`, `df.tail()` | |
| Size | `df.shape`, `df.height`, `df.width` | |
| Column types | `df.schema` (or `df.dtypes`) | also printed in every table header |
| One line per column, completeness, memory | `df.glimpse()`, `df.null_count()`, `df.estimated_size()` | |
| Summary statistics | `df.describe()` | includes `null_count` |

The check that matters after loading: **did the numbers arrive as numbers?** Read the file with
the default settings and look at the types.

In [ ]:
power_default = pl.read_csv(CSV_PATH)

text_columns = [
    name
    for name, dtype in power_default.schema.items()
    if dtype == pl.String and not name.endswith("timestamp")
]
print(len(text_columns), "numeric columns read as text (the two timestamp columns are text too):")
text_columns

Seven *numeric* columns arrived as text. Polars infers the schema from the **first 100 rows**
only (`infer_schema_length=100`); those series start later, so the first 100 rows are empty and
the reader falls back to `String`. Nothing is wrong with the data. Two fixes:

- `infer_schema_length=None`: look at the whole file. Simple, but slower;
- **declare the schema** (`schema_overrides`): fast and explicit. This is what the setup cell did.

In [ ]:
def timed(label: str, func) -> None:
    """Run a function once and print the wall-clock time."""
    tic = time.perf_counter()
    result = func()
    print(f"{label:<38} {1000 * (time.perf_counter() - tic):7.0f} ms   {result.shape}")


timed("polars, default (7 columns as text)", lambda: pl.read_csv(CSV_PATH))
timed("polars, infer_schema_length=None", lambda: pl.read_csv(CSV_PATH, infer_schema_length=None))
timed(
    "polars, schema_overrides",
    lambda: pl.read_csv(CSV_PATH, schema_overrides=schema_overrides, try_parse_dates=True),
)

In [ ]:
print(power.shape)                      # (rows, columns)
print(set(power.dtypes))                # only floats and datetimes now

power.select(cs.by_index(0, 1, 5, 6)).head(3)

In [ ]:
# glimpse(): one line per column.
power.select(cs.by_index(0, 1, 5, 6, 39)).glimpse()

In [ ]:
# describe() includes the null count; here on the tidy Great Britain table.
gb.describe()

`describe()` already hints at bad data: the mean load is about 35 GW, but the minimum is 248 MW
and the maximum 71 GW. We come back to those in sections 2.4 and 3.3.

`null_count()` answers the "how complete is each column" part of `info()`:

In [ ]:
(
    power.null_count()
    .unpivot(variable_name="column", value_name="nulls")
    .with_columns(share=(pl.col("nulls") / power.height).round(3))
    .sort("nulls", descending=True)
    .head(6)
)

### Try it 2.1: a column you have not looked at yet

`power` also holds a day-ahead electricity price: `IE_sem_price_day_ahead` (EUR/MWh).

1. How many of its values are missing, and what share of the table is that?
2. What are its minimum and maximum? Do they look like measurements?
3. Are the missing values scattered, or is part of the period simply not covered?

<details>
<summary><b>Show the solution</b></summary>

```python
PRICE = "IE_sem_price_day_ahead"

print(power.select("utc_timestamp", PRICE).describe())

power.filter(pl.col(PRICE).is_not_null()).select(
    first_valid=pl.col("utc_timestamp").min(),
    last_valid=pl.col("utc_timestamp").max(),
    n_valid=pl.len(),
)
```

- `describe()` answers the first two questions at once: 36,706 nulls out of 100,802 rows (36%),
  minimum exactly -100.0 and maximum exactly 1000.0.
- Those two round numbers are the price floor and the price cap of the Irish market of those
  years, and negative prices do happen in electricity markets. They are real values: an outlier
  filter would have deleted them.
- The valid values run from 2015-01-01 06:30 to 2018-09-28 23:00 UTC. The nulls are not random
  gaps: the series stops two years before the end of the file. A null *count* alone does not
  tell you that; where the nulls are does.

</details>

In [ ]:
# Your code here

### The same check for every column

Try it 2.1 asks where the valid values of *one* column start and end. For all 39 series at once,
reshape first: `unpivot` (section 2.6) stacks the float columns into one long table, with one
row per timestamp and column, and a `group_by` does the rest.

In [ ]:
coverage_long = (
    power.unpivot(cs.float(), index="utc_timestamp", variable_name="column")
    .drop_nulls("value")                          # keep the valid samples only
    .group_by("column", maintain_order=True)
    .agg(
        first_valid=pl.col("utc_timestamp").min(),
        last_valid=pl.col("utc_timestamp").max(),
        n_valid=pl.len(),
    )
)
print("same table as in section 1.3:", coverage_long.equals(coverage))

late = coverage_long.filter(pl.col("first_valid") > power["utc_timestamp"][99])
print(late.height, "series start after the first 100 rows:", sorted(late["column"]) == sorted(text_columns))
coverage_long.sort("first_valid", descending=True).head(8)

The series that start after row 100 are exactly the seven columns that the default reader
returned as text at the top of this section: the table explains the symptom.

Two routes to one table, with different costs. This one is the shortest to write, but it builds
a long intermediate table of 100,802 x 39 = 3.9 million rows before aggregating, and a column
with no valid value at all would vanish from the result (`drop_nulls` removes all its rows). The
route of section 1.3 aggregates first and reshapes 39 values; an empty column would stay, with
nulls. On this table both take a fraction of a second: prefer the one that reads better, and
remember the difference when the table is large.

## 2.2 Converting and handling dates

| Task | Polars | Note |
|------|--------|------|
| Text to datetime | `pl.col("s").str.to_datetime()` | also `str.to_date()`, `str.to_time()` |
| Give the format | `format="%d/%m/%Y"` | strftime codes |
| Day-first text | an explicit `format` | there is no `dayfirst` switch |
| Values that do not parse | `strict=False` | they become `null` |
| Time zone | `time_zone="UTC"`, `.dt.convert_time_zone(tz)` | one time zone per column |
| Same clock, another zone | `.dt.replace_time_zone(tz)` | |
| Components | `.dt.hour()`, `.dt.date()`, `.dt.weekday()` | methods, with parentheses; **Monday = 1** |
| Round down | `.dt.truncate("1h")` | |

### Parsing text into datetimes

In [ ]:
raw_times = pl.read_csv(CSV_PATH, columns=["utc_timestamp", "cet_cest_timestamp"])
print(raw_times.head(2))

parsed = raw_times.select(
    utc=pl.col("utc_timestamp").str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_zone="UTC"),
    cet_cest=pl.col("cet_cest_timestamp").str.to_datetime("%Y-%m-%dT%H:%M:%S%z"),
)
parsed.head(2)

Two things to notice.

- A column holds **one time zone**. The second column mixes `+0100` and `+0200` offsets (winter
  and summer time), so every value is converted to UTC: the two parsed columns are identical.
- The local wall-clock time is one conversion away, and it handles daylight saving time.

In [ ]:
local = parsed.select(
    "utc",
    berlin=pl.col("utc").dt.convert_time_zone("Europe/Berlin"),
    london=pl.col("utc").dt.convert_time_zone("Europe/London"),
    naive=pl.col("utc").dt.replace_time_zone(None),      # drop the zone, keep the clock time
)

# The last Sunday of March 2015: clocks jump from 02:00 to 03:00 in Berlin.
local.filter(pl.col("utc").is_between(datetime(2015, 3, 29, 0, 0, tzinfo=UTC), datetime(2015, 3, 29, 1, 30, tzinfo=UTC)))

### Ambiguous formats

A small "logger export" built from the real data, with timestamps written day first:
`03/04/2020` is the 3rd of April. It is also deliberately messy (duplicated and out-of-order
rows, text markers in the numeric column); section 2.3 cleans it.

In [ ]:
LOGGER_PATH = DATA_DIR / "logger_export.csv"


def write_logger_export(source: pl.DataFrame, path: Path) -> None:
    """Write a small, deliberately messy CSV extracted from the real load series."""
    rows = (
        source
        .filter(pl.col("timestamp").dt.date() == date(2020, 4, 3))
        .head(10)
        .select(
            pl.col("timestamp").dt.strftime("%d/%m/%Y %H:%M"),
            pl.col("load_mw").cast(pl.String),
        )
        .rows()
    )
    rows[4] = (rows[4][0], "ERR")                 # a failed reading
    rows[8] = (rows[8][0], "---")                 # another marker for "no value"
    messy = rows[:3] + [rows[2]] + rows[3:5] + [rows[6], rows[5]] + rows[7:] + [rows[7]]
    pl.DataFrame(messy, schema=["timestamp", "load_mw"], orient="row").write_csv(path)


write_logger_export(gb, LOGGER_PATH)
logger_raw = pl.read_csv(LOGGER_PATH)
logger_raw

In [ ]:
first = logger_raw["timestamp"][0]

guess_pandas = pd.to_datetime(pd.Series([first]))[0]
guess_polars = pl.Series([first]).str.to_datetime()[0]
explicit = pl.Series([first]).str.to_datetime("%d/%m/%Y %H:%M")[0]

print(f"text            : {first}")
print(f"pandas, guessing: {guess_pandas:%d %B %Y}")
print(f"polars, guessing: {guess_polars:%d %B %Y}")
print(f"explicit format : {explicit:%d %B %Y}")

pandas appears here for one reason: both libraries guess when no format is given, **in
opposite directions and without a warning**. pandas reads month first, Polars day first. Neither is "right": the file does not say. Polars has
no `dayfirst` switch; the fix is the same in both libraries and it is to **always pass the
format**.

`strict=False` is the counterpart of `errors="coerce"`: values that do not match become `null`
instead of raising.

In [ ]:
mixed = pl.DataFrame({"when": ["03/04/2020 10:30", "13/04/2020 11:00", "not a date"]})

try:
    mixed.select(pl.col("when").str.to_datetime("%d/%m/%Y %H:%M"))
except pl.exceptions.InvalidOperationError as err:
    print("InvalidOperationError:", str(err).splitlines()[0])

mixed.with_columns(parsed=pl.col("when").str.to_datetime("%d/%m/%Y %H:%M", strict=False))

### The `.dt` namespace

Time components for filtering by shift, weekday and so on. Mind the weekday numbering: Polars
follows ISO (Monday = 1 ... Sunday = 7).

In [ ]:
calendar = gb.select(
    "timestamp",
    hour=pl.col("timestamp").dt.hour(),
    weekday=pl.col("timestamp").dt.weekday(),
    day=pl.col("timestamp").dt.date(),
    month=pl.col("timestamp").dt.month(),
    local_hour=pl.col("timestamp").dt.convert_time_zone("Europe/London").dt.hour(),
)
row = 30_000
calendar.slice(row, 1)

In [ ]:
# Typical use: the evening peak (17:00-20:00 local time) on working days versus weekends.
london = pl.col("timestamp").dt.convert_time_zone("Europe/London")

(
    gb
    .filter(london.dt.hour().is_between(17, 19))
    .group_by(is_weekend=london.dt.weekday() >= 6)
    .agg(n=pl.len(), evening_load_mean=pl.col("load_mw").mean().round(0))
    .sort("is_weekend")
)

### Try it 2.2: load by day of the week

Compute the mean Great Britain load (`gb`, column `load_mw`) for each day of the week, using the
**local** calendar (`Europe/London`). Which day has the lowest mean load, and which number does
Polars give to that day?

<details>
<summary><b>Show the solution</b></summary>

```python
london_time = pl.col("timestamp").dt.convert_time_zone("Europe/London")

(
    gb
    .group_by(weekday=london_time.dt.weekday())
    .agg(load_mean=pl.col("load_mw").mean().round(0))
    .sort("weekday")
)
```

Sunday is the lowest, 32,166 MW, against about 36,100 MW from Tuesday to Thursday. Polars calls
Sunday **7** (ISO numbering, Monday = 1).

Without `convert_time_zone` the grouping uses the UTC calendar: the numbers change a little
(Sunday becomes 32,144 MW), because in summer the UTC day starts at 01:00 local time.

</details>

In [ ]:
# Your code here

## 2.3 Cleaning data

| Task | Polars | Note |
|------|--------|------|
| Test for missing values | `is_null()`, `is_not_null()` | plus `is_nan()` for floats |
| Drop rows with missing values | `drop_nulls()` | |
| Fill with a value | `fill_null(value)` | |
| Hold the last or the next value | `forward_fill()`, `backward_fill()` | both take `limit=` |
| Interpolate by row position | `interpolate()` | linear |
| Interpolate by elapsed time | `interpolate_by("timestamp")` | for irregular sampling |
| Duplicates | `is_duplicated()`, `unique()` | |
| Put rows in time order | `sort("timestamp")` | there is no index: sort by the column |
| Change type | `cast(pl.Float64)` | strict: raises on failure |
| Text with bad values to numbers | `cast(pl.Float64, strict=False)` | or `null_values=[...]` when reading |

### First, the one real difference: `null` is not `NaN`

In pandas `NaN` is the missing-value marker and aggregations skip it. Polars separates the two:

| | `null` | `NaN` |
|---|--------|-------|
| Meaning | the value is **missing** | a floating-point value that is **not a number** (0/0) |
| Available for | every data type | floats only |
| Aggregations | **skipped** | **propagated**: the result is `NaN` |
| Test / fill | `is_null()` / `fill_null()` | `is_nan()` / `fill_nan()` |

In [ ]:
values = [1.0, None, float("nan"), 4.0]

print("pandas mean:", pd.Series(values).mean(), " (None and NaN are both 'missing')")

pl.DataFrame({"x": values}).select(
    mean_raw=pl.col("x").mean(),                     # the NaN propagates
    mean_clean=pl.col("x").fill_nan(None).mean(),    # nulls are skipped: (1 + 4) / 2
    n_null=pl.col("x").null_count(),
    n_valid=pl.col("x").count(),                     # count() excludes nulls only
)

Rule of thumb: data read from CSV or Parquet arrives with `null` for missing values. Data built
from NumPy arrays or converted from pandas may carry `NaN`: convert once, on the way in, with
`fill_nan(None)`.

### Missing values: find, drop, fill

Sensor dropouts are normal. Here is a real one in the wind series: 12 missing half-hours on
26 August 2017.

In [ ]:
print(gb.null_count())

gap = gb.filter(
    pl.col("timestamp").is_between(datetime(2017, 8, 26, 3, 0, tzinfo=UTC), datetime(2017, 8, 26, 11, 0, tzinfo=UTC))
)

gap.select(
    "timestamp",
    "wind_mw",
    is_null=pl.col("wind_mw").is_null(),
    forward=pl.col("wind_mw").forward_fill(),
    forward_max_2=pl.col("wind_mw").forward_fill(limit=2),
    backward=pl.col("wind_mw").backward_fill(),
    constant=pl.col("wind_mw").fill_null(0.0),
    linear=pl.col("wind_mw").interpolate().round(1),
).slice(1, 6)

Forward fill is often physically meaningful (the last known value still holds); `limit` stops it
from inventing hours of data. Dropping incomplete rows is explicit:

In [ ]:
print("all rows                 :", gb.height)
print("load present             :", gb.drop_nulls("load_mw").height)
print("every column present     :", gb.drop_nulls().height)

### Interpolation and irregular sampling

`interpolate()` is linear **by row position**. With irregular sampling that is wrong:
`interpolate_by("timestamp")` fills proportionally to the time elapsed.

In [ ]:
irregular = pl.DataFrame(
    {
        "timestamp": [
            datetime(2020, 1, 1, 0, 0),
            datetime(2020, 1, 1, 0, 30),    # 30 minutes later
            datetime(2020, 1, 1, 3, 0),     # 2.5 hours later
            datetime(2020, 1, 1, 4, 0),
        ],
        "x": [0.0, None, None, 40.0],
    }
)

irregular.with_columns(
    by_row=pl.col("x").interpolate(),
    by_time=pl.col("x").interpolate_by("timestamp"),
)

### Duplicates, order and polluted columns

Back to the logger export: duplicated rows, two rows out of order, and `ERR` / `---` in a numeric
column, which is why the whole column arrived as text.

In [ ]:
try:
    logger_raw.with_columns(pl.col("load_mw").cast(pl.Float64))
except pl.exceptions.InvalidOperationError as err:
    print("InvalidOperationError:", str(err).splitlines()[0])

logger = logger_raw.with_columns(
    pl.col("timestamp").str.to_datetime("%d/%m/%Y %H:%M", time_zone="UTC"),
    pl.col("load_mw").cast(pl.Float64, strict=False),      # bad values become null
)
logger

In [ ]:
print("sorted by time?      ", logger["timestamp"].is_sorted())
print("rows that have a twin:", logger.is_duplicated().sum())

logger.filter(pl.col("timestamp").is_duplicated())

In [ ]:
logger_clean = (
    logger
    .unique(subset="timestamp", keep="first", maintain_order=True)   # drop_duplicates
    .sort("timestamp")                                               # sort_index
)
print(logger_clean["timestamp"].is_sorted(), logger_clean["timestamp"].is_unique().all())
logger_clean

When the markers are known in advance, the cleanest fix is at read time: tell the reader which
strings mean "missing" and the column is numeric from the start.

In [ ]:
pl.read_csv(LOGGER_PATH, null_values=["ERR", "---"]).schema

### Try it 2.3: three ways to fill the same gaps

`load_mw` in `gb` has 16 missing values. In a single `select`, count how many are still missing
after:

1. a forward fill limited to 2 samples (one hour);
2. a forward fill without limit;
3. a linear interpolation.

Neither 2 nor 3 gets to zero, and they do not agree with each other. Why?

<details>
<summary><b>Show the solution</b></summary>

```python
print(
    gb.select(
        missing=pl.col("load_mw").null_count(),
        forward_limit_2=pl.col("load_mw").forward_fill(limit=2).null_count(),
        forward=pl.col("load_mw").forward_fill().null_count(),
        linear=pl.col("load_mw").interpolate().null_count(),
    )
)

# Where the missing values are:
gb.with_row_index().filter(pl.col("load_mw").is_null()).select("index", "timestamp")
```

16 missing, then 11, 3 and 4.

- The 16 nulls are the first three rows, two gaps of six samples (the daylight-saving nights of
  2015 and 2017) and the very last row.
- `limit=2` fills two samples of each six-sample gap and the last row: 16 - 5 = 11.
- An unlimited forward fill cannot fill the first three rows: there is no earlier value to copy.
- Interpolation needs a value on **both** sides, so it leaves the first three rows and the last
  one. Neither method extrapolates backwards; `backward_fill()` would.

</details>

In [ ]:
# Your code here

## 2.4 Basic operations

| Task | Polars | Note |
|------|--------|------|
| Compute a new column | `df.with_columns(p=pl.col("v") * pl.col("i"))` | vectorised, no loops |
| Combine two tables row by row | an explicit `join` | nothing is aligned for you |
| Rename columns | `rename({...})` | |
| Drop columns | `drop(...)` | |
| Statistics | `sum`, `mean`, `min`, `max`, `std`, `quantile`, as expressions | nulls are skipped |
| Plot | matplotlib, or plotly express for an interactive chart | `px.line(df, x=..., y=...)` |

### Vectorised arithmetic

In [ ]:
gb_derived = (
    gb
    .with_columns(renewables_mw=pl.col("wind_mw") + pl.col("solar_mw"))
    .with_columns(
        renewable_share=pl.col("renewables_mw") / pl.col("load_mw"),
        forecast_error_mw=pl.col("load_forecast_mw") - pl.col("load_mw"),
    )
)
gb_derived.select("timestamp", "load_mw", "renewables_mw", "renewable_share", "forecast_error_mw").slice(60_000, 3)

### No automatic alignment

pandas aligns two objects on their index labels before computing; labels present on one side
only give `NaN`. Polars has no index, so nothing is aligned for you: columns of the same frame
combine row by row, and two *different* tables are brought together with an explicit `join`.

In [ ]:
a_pd = pd.Series([1.0, 2.0, 3.0], index=[0, 1, 2])
b_pd = pd.Series([10.0, 20.0, 30.0], index=[1, 2, 3])
print("pandas, aligned on the index:", (a_pd + b_pd).to_dict())

a = pl.DataFrame({"t": [0, 1, 2], "a": [1.0, 2.0, 3.0]})
b = pl.DataFrame({"t": [1, 2, 3], "b": [10.0, 20.0, 30.0]})

a.join(b, on="t", how="full", coalesce=True).sort("t").with_columns(total=pl.col("a") + pl.col("b"))

### Everyday column management

In [ ]:
tidy = (
    gb_derived
    .rename({"load_mw": "demand_mw"})
    .drop("load_forecast_mw", "forecast_error_mw")
)
print(tidy.columns)

# Reordering is a select; selectors help to say "and then everything else".
print(tidy.select("timestamp", "renewable_share", cs.ends_with("_mw")).columns)

### Basic statistics

Aggregations are expressions, so several of them run in one `select`, over one or many columns.

In [ ]:
gb.select(
    total=pl.col("load_mw").sum(),
    mean=pl.col("load_mw").mean(),
    std=pl.col("load_mw").std(),
    minimum=pl.col("load_mw").min(),
    p05=pl.col("load_mw").quantile(0.05),
    median=pl.col("load_mw").median(),
    p95=pl.col("load_mw").quantile(0.95),
    maximum=pl.col("load_mw").max(),
)

In [ ]:
# The same statistics for every numeric column, and a correlation.
print(gb.select(cs.numeric().mean().round(1).name.suffix("_mean")))
print(gb.select(forecast_vs_actual=pl.corr("load_forecast_mw", "load_mw")))

### Quick visual checks

A plot is still the fastest way to spot bad data. matplotlib accepts Polars columns directly.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(gb["timestamp"], gb["load_mw"] / 1000, linewidth=0.4, color="tab:blue")
ax.set_ylabel("load [GW]")
ax.set_title("Great Britain, actual load, 30-minute values")
ax.grid(alpha=0.3)
plt.show()

The seasonal pattern is clear, and so are the problems `describe()` hinted at: one spike to
71 GW and a number of values close to zero. They are in the source data. The suspicious rows:

In [ ]:
suspicious = gb.filter((pl.col("load_mw") < 15_000) | (pl.col("load_mw") > 60_000))
print(suspicious.height, "suspicious half-hours")

suspicious.group_by(year=pl.col("timestamp").dt.year()).agg(
    n=pl.len(),
    lowest=pl.col("load_mw").min(),
    highest=pl.col("load_mw").max(),
).sort("year")

For an **interactive** chart (zoom, pan, values on hover) plotly express takes a Polars frame
directly; Colab has plotly installed. Here the daily mean: 2,101 points are plenty for a first
look, and a small figure keeps the notebook light.

In [ ]:
daily_load = (
    gb.sort("timestamp")
    .group_by_dynamic("timestamp", every="1d")
    .agg(load_gw=pl.col("load_mw").mean() / 1000)
)
print(daily_load.height, "rows")

fig = px.line(daily_load, x="timestamp", y="load_gw", title="Great Britain: daily mean load (GW)")
fig.update_layout(height=320, margin={"l": 40, "r": 20, "t": 50, "b": 40})
fig.show()

### Try it 2.4: residual load

The *residual load* is the demand left for conventional plants once wind and solar are
subtracted: `load_mw - wind_mw - solar_mw`.

1. Add it to `gb` as `residual_load_mw` (do not overwrite `gb`: give the result a new name).
2. Compute its minimum and count the half-hours in which it is negative.
3. Look at the three rows with the lowest residual load. Would you trust them?

<details>
<summary><b>Show the solution</b></summary>

```python
residual = gb.with_columns(
    residual_load_mw=pl.col("load_mw") - pl.col("wind_mw") - pl.col("solar_mw")
)

print(
    residual.select(
        minimum=pl.col("residual_load_mw").min(),
        negative=(pl.col("residual_load_mw") < 0).sum(),
    )
)

residual.bottom_k(3, by="residual_load_mw")
```

The minimum is -9,686 MW and 143 half-hours are negative. Do not trust them: in the three
lowest rows the load is about 3 GW while the forecast for the same half-hour is 29 to 39 GW.
These are the bad load samples of the plot above, and a derived column inherits them.

If you keep only plausible loads (`.filter(pl.col("load_mw") > 15_000)`) the minimum becomes
3,954 MW, on 23 May 2020 at 13:00 UTC: a windy, sunny Saturday with low demand.

</details>

In [ ]:
# Your code here

## 2.5 Combining data

| Task | Polars | Note |
|------|--------|------|
| Stack tables | `pl.concat([a, b])` | `how="vertical"` (default), `"diagonal"`, `"horizontal"` |
| Read many files as one table | `pl.read_csv("folder/*.csv")` | the reader accepts a glob pattern |
| Join on a key | `a.join(b, on=..., how=...)` | `how`: inner, left, right, full, semi, anti, cross |
| Check the relationship | `validate="m:1"` | raises if the data disagree |
| Join on the nearest timestamp | `a.join_asof(b, ...)` | |

### Stacking files: `concat`

A very common real-world task: one file per month. We first write 2019 as twelve CSV files.

In [ ]:
MONTHLY_DIR = DATA_DIR / "monthly"
MONTHLY_DIR.mkdir(exist_ok=True)

gb_2019 = gb.filter(pl.col("timestamp").dt.year() == 2019)
for (month,), part in gb_2019.group_by(pl.col("timestamp").dt.month(), maintain_order=True):
    part.write_csv(MONTHLY_DIR / f"gb_2019_{month:02d}.csv")

files = sorted(MONTHLY_DIR.glob("gb_2019_*.csv"))
print(len(files), "files:", files[0].name, "...", files[-1].name)

In [ ]:
# By hand: read each file, then concatenate.
stacked = pl.concat([pl.read_csv(file, try_parse_dates=True) for file in files])

# Better: let the reader expand the pattern (and keep track of the source file).
globbed = pl.read_csv(
    MONTHLY_DIR / "gb_2019_*.csv", try_parse_dates=True, include_file_paths="source_file"
)

print(stacked.shape, globbed.shape, "| identical to the original:", stacked.equals(gb_2019))
globbed.group_by("source_file").agg(rows=pl.len()).sort("source_file").head(3)

`concat` is strict: the frames must have the same columns and types. `how="diagonal"` takes the
union of the columns and fills the gaps with `null`.

In [ ]:
january = pl.DataFrame({"timestamp": [date(2019, 1, 1)], "load_mw": [30_000.0]})
february = pl.DataFrame({"timestamp": [date(2019, 2, 1)], "load_mw": [31_000.0], "note": ["estimated"]})

try:
    pl.concat([january, february])
except pl.exceptions.ShapeError as err:
    print("ShapeError:", str(err).splitlines()[0])

pl.concat([january, february], how="diagonal")

### Measurements plus metadata: `join`

The long `regional` table has a region code; the names live in the small `regions` table.
`validate="m:1"` states the relationship we expect (many readings, one region) and raises if the
data disagree.

In [ ]:
regions

In [ ]:
regional_named = regional.join(regions, on="region", how="left", validate="m:1")

(
    regional_named
    .group_by("country", "name")
    .agg(load_mean_mw=pl.col("load_mw").mean().round(0), load_max_mw=pl.col("load_mw").max())
    .sort("load_mean_mw", descending=True)
)

`semi` and `anti` joins filter one table by the keys of another, without adding columns.

In [ ]:
island_of_ireland = pl.DataFrame({"region": ["IE", "GB_NIR"]})

print(regions.join(island_of_ireland, on="region", how="semi")["name"].to_list())
print(regions.join(island_of_ireland, on="region", how="anti")["name"].to_list())

### Joining on the nearest timestamp: `join_asof`

Events are rarely logged exactly on the measurement grid. `join_asof` attaches to each row of
the left table the closest row of the right table in time: `strategy` is `"backward"` (last
reading at or before the event), `"forward"` or `"nearest"`, and `tolerance` limits how far to
look. Both tables must be sorted by the time key.

In [ ]:
events = pl.DataFrame(
    {
        "event_time": [
            datetime(2019, 1, 15, 18, 7, tzinfo=UTC),
            datetime(2019, 6, 2, 3, 44, tzinfo=UTC),
            datetime(2020, 3, 29, 1, 59, tzinfo=UTC),
        ],
        "event": ["evening peak alarm", "night maintenance", "clock change"],
    }
)

events.join_asof(
    gb.select("timestamp", "load_mw"),
    left_on="event_time",
    right_on="timestamp",
    strategy="backward",
    tolerance="30m",
)

### Try it 2.5: filter one table with another

Build `uk_regions`, the rows of `regions` whose `country` is `"United Kingdom"`. Then keep only
the rows of `regional` that belong to those regions, **without adding any column**.

1. Which kind of join does that?
2. How many rows and columns does the result have, and how many valid load readings per region?
3. What does the same join with `how="anti"` return?

<details>
<summary><b>Show the solution</b></summary>

```python
uk_regions = regions.filter(pl.col("country") == "United Kingdom")

uk_rows = regional.join(uk_regions, on="region", how="semi")
print(uk_rows.shape, uk_rows.columns)

uk_rows.group_by("region").agg(n_valid=pl.col("load_mw").count()).sort("region")
```

A **semi** join: 201,604 rows and still the 4 columns of `regional`, with 100,786 valid
readings for `GB_GBN` and 100,362 for `GB_NIR`. An inner join would keep the same rows but add
`name` and `country`.

`how="anti"` is the complement: the rows of `regional` whose region is *not* in `uk_regions`
(`IE` and `CY`). It is the quick way to answer "which keys have no match in the other table?".

</details>

In [ ]:
# Your code here

## 2.6 Grouping and reshaping

| Task | Polars | Note |
|------|--------|------|
| One row per group | `group_by(keys).agg(...)` | keys stay ordinary columns; group order is not defined |
| A group statistic on every row | `pl.col(col).mean().over(keys)` | keeps every row |
| Long to wide | `pivot(on, index=, values=)` | eager only |
| Long to wide, repeated keys | `pivot(..., aggregate_function=)` | |
| Wide to long | `unpivot(index=, ...)` | |

### `group_by(...).agg(...)`: one row per group

In [ ]:
(
    regional
    .group_by("region")
    .agg(
        n_valid=pl.col("load_mw").count(),
        load_mean=pl.col("load_mw").mean().round(1),
        load_p95=pl.col("load_mw").quantile(0.95).round(1),
        wind_share_mean=(pl.col("wind_onshore_mw") / pl.col("load_mw")).mean().round(3),
    )
    .sort("load_mean", descending=True)
)

In [ ]:
# Group keys can be expressions: the average daily profile, working days versus weekends.
london = pl.col("timestamp").dt.convert_time_zone("Europe/London")

profile = (
    gb
    .group_by(hour=london.dt.hour(), is_weekend=london.dt.weekday() >= 6)
    .agg(load_mean_gw=(pl.col("load_mw").mean() / 1000).round(2))
    .sort("hour", "is_weekend")
)
profile.filter(pl.col("hour").is_in([4, 8, 18]))

### `.over()`: a group statistic on every row

`group_by().agg()` collapses each group to one row. `.over()` computes the same statistic but
**keeps every row**, like `groupby().transform()` in pandas or a window function in SQL.

In [ ]:
(
    regional
    .select(
        "timestamp",
        "region",
        "load_mw",
        region_mean=pl.col("load_mw").mean().over("region").round(1),
        relative_load=(pl.col("load_mw") / pl.col("load_mw").mean().over("region")).round(3),
        rank_in_region=pl.col("load_mw").rank(descending=True).over("region"),
    )
    .filter(pl.col("timestamp") == datetime(2019, 1, 15, 18, 0, tzinfo=UTC))
)

### Long to wide and back: `pivot` and `unpivot`

`pivot` is eager only, because the output columns depend on the data.

In [ ]:
monthly_long = (
    regional.sort("region", "timestamp")
    .group_by_dynamic("timestamp", every="1mo", group_by="region")
    .agg(load_mean=pl.col("load_mw").mean().round(1))
    .sort("region", "timestamp")
)
print(monthly_long.shape)

monthly_wide = monthly_long.pivot("region", index="timestamp", values="load_mean")
monthly_wide.tail(3)

In [ ]:
monthly_wide.unpivot(index="timestamp", variable_name="region", value_name="load_mean").tail(3)

The original file is itself "wide": the region is encoded in the column names. `unpivot` plus
a string expression turns any group of those columns into a tidy table.

In [ ]:
all_loads = (
    power
    .select("utc_timestamp", cs.ends_with("load_actual_entsoe_transparency"))
    .unpivot(index="utc_timestamp", variable_name="series", value_name="load_mw")
    .with_columns(region=pl.col("series").str.extract(r"^(CY|GB_GBN|GB_NIR|GB_UKM|IE_sem|IE)_", 1))
)
all_loads.group_by("region").agg(
    n_valid=pl.col("load_mw").count(),
    load_mean=pl.col("load_mw").mean().round(1),
).sort("region")

### Try it 2.6: one row per year, one column per region

Build a table with one row per year and one column per region, containing the **highest** load
of that year (`regional`, column `load_mw`). An ordinary `group_by` on the year is enough, then
`pivot`.

Two things in the result deserve a comment: one row is empty, and one number cannot be right.
Which ones, and why?

<details>
<summary><b>Show the solution</b></summary>

```python
yearly_peak = (
    regional
    .group_by("region", year=pl.col("timestamp").dt.year())
    .agg(load_max=pl.col("load_mw").max())
    .sort("year", "region")
)

yearly_peak.pivot("region", index="year", values="load_max").sort("year")
```

- The 2014 row is empty: the file starts at 23:00 UTC on 31 December 2014, so two half-hours
  fall in 2014, and they have no load. Cyprus is also empty in 2015: its series starts later.
- Great Britain peaks at 51 to 58 GW in every year except 2018, where the table says 71,273 MW.
  That is one of the bad samples met in section 2.4: a maximum is the statistic most exposed to
  a single wrong value.

The `sort` calls are not decoration: `group_by` returns the groups in no particular order, and
`pivot` creates the columns in order of appearance.

</details>

In [ ]:
# Your code here

# Part 3 - Time series

There is no index, so every time-series tool takes the **name of the time column**, and expects
that column to be **sorted**.

In [ ]:
gb = gb.sort("timestamp")                          # already sorted; say it explicitly
print("sorted by time:", gb["timestamp"].is_sorted())

## 3.1 Resampling and frequency

| Task | Polars | Note |
|------|--------|------|
| Fixed time buckets | `group_by_dynamic("timestamp", every="1h").agg(...)` | |
| Several statistics per bucket | several expressions inside `agg` | |
| Bucket edges and labels | `closed=`, `label=` | defaults: left / left |
| One series per group | `group_by_dynamic(..., group_by=key)` | |
| Put a table on a regular grid | `upsample("timestamp", every=...)` | eager only |
| Build a range of timestamps | `pl.datetime_range(...)`, `pl.date_range(...)` | |
| Open, high, low, close | `first`, `max`, `min`, `last` inside `agg` | no shortcut |

### Downsampling

In [ ]:
hourly = gb.group_by_dynamic("timestamp", every="1h").agg(load_mw=pl.col("load_mw").mean())
print(gb.height, "->", hourly.height, "rows")
hourly.slice(1, 3)

In [ ]:
# Several aggregations at once.
daily = gb.group_by_dynamic("timestamp", every="1d").agg(
    load_mean=pl.col("load_mw").mean().round(1),
    load_min=pl.col("load_mw").min(),
    load_max=pl.col("load_mw").max(),
    n_valid=pl.col("load_mw").count(),
)
daily.slice(1, 3)

In [ ]:
# Per group: add group_by. The frame must be sorted by time within each group.
(
    regional.sort("region", "timestamp")
    .group_by_dynamic("timestamp", every="1y", group_by="region")
    .agg(load_mean=pl.col("load_mw").mean().round(1))
    .filter(pl.col("timestamp").dt.year() == 2019)
    .sort("region")
)

### `label` and `closed`

Which samples fall in a bucket (`closed`), and is the bucket named by its start or by its end
(`label`)? For metered energy this matters: a value stamped 01:00 may describe the half-hour
*ending* at 01:00. Six consecutive half-hourly samples, grouped by hour in the four ways:

In [ ]:
six = gb.slice(3, 6).select("timestamp", "load_mw")
print(six["timestamp"].dt.strftime("%H:%M").to_list())

for closed, label in [("left", "left"), ("left", "right"), ("right", "left"), ("right", "right")]:
    buckets = six.group_by_dynamic("timestamp", every="1h", closed=closed, label=label).agg(
        samples=pl.col("timestamp").dt.strftime("%H:%M"),
    )
    print(f"\nclosed={closed!r}, label={label!r}")
    for bucket, samples in buckets.select(pl.col("timestamp").dt.strftime("%H:%M"), "samples").rows():
        print(f"  {bucket}  <-  {samples}")

### Regular grids: `upsample`, and making gaps explicit

A logger that simply skips rows when it has nothing to report produces an irregular table where
gaps are invisible. `upsample` puts the table on a regular grid: the missing rows appear, filled
with `null`.

In [ ]:
march_29 = pl.col("timestamp").is_between(
    datetime(2015, 3, 29, 0, 0, tzinfo=UTC), datetime(2015, 3, 29, 4, 30, tzinfo=UTC)
)
with_holes = gb.filter(march_29).select("timestamp", "load_mw").drop_nulls()   # rows are simply absent
print(with_holes["timestamp"].dt.strftime("%H:%M").to_list())

on_grid = with_holes.upsample("timestamp", every="30m")
print(with_holes.height, "->", on_grid.height, "rows,", on_grid["load_mw"].null_count(), "explicit gaps")
on_grid

That gap is real and has a classic cause: it is the night of the switch to summer time
(29 March 2015), and the source lost three hours of British load.

Upsampling to a finer grid works the same way; then choose how to fill.

In [ ]:
(
    gb.slice(60, 3).select("timestamp", "load_mw")
    .upsample("timestamp", every="15m")
    .with_columns(
        held=pl.col("load_mw").forward_fill(),
        linear=pl.col("load_mw").interpolate(),
    )
)

### Ranges and duration strings

Polars uses one small language for durations everywhere (`every`, `period`, `window_size`,
`interval`): a number followed by a unit, and units can be combined (`"1h30m"`).

| Meaning | Polars |
|---------|--------|
| second, minute, hour | `"1s"`, `"1m"`, `"1h"` |
| day, week | `"1d"`, `"1w"` |
| month, quarter, year | `"1mo"`, `"1q"`, `"1y"` |

Mind the trap: in Polars `"1m"` is one **minute** and `"1mo"` one month. Calendar windows in
Polars are labelled by their start, so there is no month-start / month-end pair of aliases.

In [ ]:
half_hours = pl.datetime_range(
    datetime(2020, 1, 1), datetime(2020, 1, 1, 2), interval="30m", time_zone="UTC", eager=True
)
months = pl.date_range(date(2020, 1, 1), date(2020, 6, 1), interval="1mo", eager=True)

print(half_hours.to_list()[:3])
print(months.to_list())

# A complete grid is also the direct way to check for missing rows.
grid = pl.DataFrame(
    {"timestamp": pl.datetime_range(gb["timestamp"].min(), gb["timestamp"].max(), interval="30m", eager=True)}
)
print("rows missing from the 30-minute grid:", grid.join(gb, on="timestamp", how="anti").height)

### Open, high, low, close

Useful to summarise any fluctuating signal per interval. Polars has no `ohlc()` shortcut: it is
four expressions. One thing to know: `first()` and `last()` return the first and last **rows**,
nulls included. Add `drop_nulls()` to get the first and last valid values.

In [ ]:
ohlc = gb.group_by_dynamic("timestamp", every="1d").agg(
    open=pl.col("load_mw").drop_nulls().first(),
    high=pl.col("load_mw").max(),
    low=pl.col("load_mw").min(),
    close=pl.col("load_mw").drop_nulls().last(),
)
ohlc.slice(1, 2)

### Try it 3.1: weekly means

Extract the top 3 weeks for **weekly** mean in GB load (hint: use `every="1w"`), as well as the number of valid
samples of each week next to the mean.

Bonus:

1. On which day of the week do the windows start?
2. Which week has the highest mean load?
3. A full week has 7 x 48 = 336 samples. Which weeks have fewer, and why?


**Expected result**

```raw
shape: (3, 3)
┌─────────────────────────┬───────────┬─────────┐
│ timestamp               ┆ load_mean ┆ n_valid │
│ ---                     ┆ ---       ┆ ---     │
│ datetime[μs, UTC]       ┆ f64       ┆ u32     │
╞═════════════════════════╪═══════════╪═════════╡
│ 2015-01-19 00:00:00 UTC ┆ 44667.0   ┆ 336     │
│ 2015-02-02 00:00:00 UTC ┆ 44567.0   ┆ 336     │
│ 2016-01-11 00:00:00 UTC ┆ 44332.0   ┆ 336     │
└─────────────────────────┴───────────┴─────────┘

```


<details>
<summary><b>Show the solution</b></summary>

```python
weekly_load = gb.group_by_dynamic("timestamp", every="1w").agg(
    load_mean=pl.col("load_mw").mean().round(0),
    n_valid=pl.col("load_mw").count(),
)
print(weekly_load.height, "weeks; the first window starts on weekday", weekly_load["timestamp"].dt.weekday()[0])
print(weekly_load.top_k(3, by="load_mean"))

weekly_load.filter(pl.col("n_valid") < 336)
```

- 301 weekly windows, and they start on **Monday** (weekday 1) at 00:00 UTC.
- The highest is the week of 19 January 2015: 44,667 MW on average.
- Four weeks are incomplete: the first (191 samples) and the last (143) because the file starts
  and ends in the middle of a week, and the weeks of 23 March 2015 and 20 March 2017 (330
  samples) because of the six-sample gaps of section 2.3.

A mean over 143 samples and one over 336 look the same in the result. Carrying `n_valid` along
is what lets you tell them apart, or filter the partial windows out.

</details>

In [ ]:
# Your code here

## 3.2 Window functions

| Task | Polars | Note |
|------|--------|------|
| Window counted in rows | `rolling_mean(window_size=48)` | |
| Window measured in time | `rolling_mean_by("timestamp", window_size="1d")` | needs a sorted time column |
| Centred window | `center=True` | row-based windows only |
| Fewer samples than the window | `min_samples=` | |
| Several statistics at once | `df.rolling("timestamp", period=...).agg(...)` | |
| Exponential smoothing | `ewm_mean(span=...)` | also `ewm_mean_by(..., half_life=)` in time |
| Running statistics | `cum_max()`, `cum_sum()`, `cum_min()`, `cum_count()` | |

`rolling_*` exists for `mean`, `sum`, `min`, `max`, `std`, `var`, `median`, `quantile`.

### Rolling windows: rows versus time

In [ ]:
rolled = gb.select(
    "timestamp",
    "load_mw",
    rows_48=pl.col("load_mw").rolling_mean(window_size=48),
    time_1d=pl.col("load_mw").rolling_mean_by("timestamp", window_size="1d"),
    centered=pl.col("load_mw").rolling_mean(window_size=48, center=True),
    lenient=pl.col("load_mw").rolling_mean(window_size=48, min_samples=1),
)
print(rolled.slice(3, 2))      # start of the series: not enough samples yet
print(rolled.slice(60, 2))

On a regular, complete grid the two coincide (48 half-hours are one day). They differ as soon as
samples are missing or irregular: the row window silently stretches over more time, the time
window does not. **With real sensor data prefer the time-based form.** Row-based windows also
return `null` until the window is full (`min_samples` relaxes that); time-based ones need a
single sample by default.

For several statistics over the same window there is a `rolling` context, the sibling of
`group_by_dynamic`:

In [ ]:
(
    gb.rolling("timestamp", period="1d")
    .agg(
        load_mean=pl.col("load_mw").mean().round(1),
        load_std=pl.col("load_mw").std().round(1),
        load_max=pl.col("load_mw").max(),
        n=pl.col("load_mw").count(),
    )
    .slice(60, 3)
)

### Exponential smoothing

`ewm_mean` behaves like a first-order low-pass filter: each output is a weighted mean of the
past, with weights decaying exponentially. `span`, `half_life`, `com` and `alpha` are four ways
to give the same decay. `ewm_mean_by` takes the half-life as a duration and uses the
real timestamps, for irregular sampling.

In [ ]:
smoothed = gb.select(
    "timestamp",
    "load_mw",
    ewm_span_48=pl.col("load_mw").ewm_mean(span=48),
    ewm_6h=pl.col("load_mw").ewm_mean_by("timestamp", half_life="6h"),
)
smoothed.slice(60, 2)

### Cumulative ("expanding") statistics

Running totals and running extremes. The integral of power over time is energy: each half-hour
at `P` MW is `P / 2` MWh.

In [ ]:
gb.select(
    "timestamp",
    "load_mw",
    running_peak=pl.col("load_mw").cum_max(),
    energy_gwh=(pl.col("load_mw") * 0.5).cum_sum() / 1000,
    running_mean=(pl.col("load_mw").cum_sum() / pl.col("load_mw").cum_count()).round(1),
).slice(60, 3)

In [ ]:
# Compute on the whole series, then cut the week to plot: windows need their history.
week = gb.with_columns(
    rolling_1d=pl.col("load_mw").rolling_mean_by("timestamp", window_size="1d"),
    ewm_6h=pl.col("load_mw").ewm_mean_by("timestamp", half_life="6h"),
).filter(
    pl.col("timestamp").is_between(datetime(2019, 1, 14, tzinfo=UTC), datetime(2019, 1, 21, tzinfo=UTC))
)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(week["timestamp"], week["load_mw"] / 1000, color="0.7", linewidth=1.0, label="30-minute load")
ax.plot(week["timestamp"], week["rolling_1d"] / 1000, color="tab:blue", linewidth=1.8, label="rolling mean, 1 day")
ax.plot(week["timestamp"], week["ewm_6h"] / 1000, color="tab:orange", linewidth=1.8, label="exponential, half-life 6 h")
ax.set_ylabel("load [GW]")
ax.set_title("Great Britain, one week in January 2019")
ax.set_ylim(top=56)
ax.legend(frameon=False, ncol=3, loc="upper center")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

### Try it 3.2: a short and a long window

For the Great Britain load, compute three time-based rolling statistics: the mean over the last
**3 hours**, the mean over the last **7 days** and the maximum over the last 7 days.

Look at 15 January 2019, 18:00 UTC. How far above the 7-day mean is the load at that moment?
When was the 7-day maximum reached?

<details>
<summary><b>Show the solution</b></summary>

```python
load_windows = gb.select(
    "timestamp",
    "load_mw",
    mean_3h=pl.col("load_mw").rolling_mean_by("timestamp", window_size="3h").round(0),
    mean_7d=pl.col("load_mw").rolling_mean_by("timestamp", window_size="7d").round(0),
    max_7d=pl.col("load_mw").rolling_max_by("timestamp", window_size="7d"),
)

load_windows.filter(
    pl.col("timestamp").is_between(
        datetime(2019, 1, 15, 17, 0, tzinfo=UTC), datetime(2019, 1, 15, 18, 30, tzinfo=UTC)
    )
)
```

At 18:00 the load is 48,825 MW, the 3-hour mean 47,758 MW and the 7-day mean 37,894 MW: the
evening peak is 29% above the weekly average. The 7-day maximum, 49,091 MW, was reached one
hour earlier, at 17:00.

The short window follows the daily cycle, the long one removes it. The window size is a
modelling choice, not a technical detail: say which one you used whenever you report a
"rolling mean".

</details>

In [ ]:
# Your code here

## 3.3 Changes and lags

| Task | Polars | Note |
|------|--------|------|
| Value n rows earlier | `shift(n)` | lag features |
| Change from the previous row | `diff()` | |
| Relative change | `pct_change()` | |
| Rate of change in time | `pl.col("x").diff() / pl.col("timestamp").diff().dt.total_seconds()` | the time is just another column |

With several series in one long table, add `.over("region")` after sorting, otherwise the last
sample of one region leaks into the first sample of the next.

In [ ]:
changes = gb.select(
    "timestamp",
    "load_mw",
    previous=pl.col("load_mw").shift(1),
    same_time_yesterday=pl.col("load_mw").shift(48),
    delta=pl.col("load_mw").diff(),
    pct=pl.col("load_mw").pct_change().round(4),
    ramp_mw_per_min=(pl.col("load_mw").diff() / pl.col("timestamp").diff().dt.total_minutes()).round(2),
)
changes.slice(60, 3)

### Detecting steps and spikes

Real demand does not change by tens of gigawatts in half an hour. Sorting by the size of the
step finds the bad values seen in the plot of section 2.4 at once: each isolated bad sample
shows up as a jump down and a jump back.

In [ ]:
(
    changes
    .with_columns(abs_delta=pl.col("delta").abs())
    .top_k(6, by="abs_delta")
    .select("timestamp", "previous", "load_mw", "delta")
)

In [ ]:
# A simple, physically motivated filter: mask samples reached by an implausible ramp,
# then fill short gaps in time. limit keeps long outages as gaps.
MAX_STEP_MW = 8_000

cleaned = gb.with_columns(
    load_clean=pl.when(pl.col("load_mw").diff().abs() > MAX_STEP_MW)
    .then(None)
    .otherwise(pl.col("load_mw"))
).with_columns(
    load_clean=pl.when(pl.col("load_clean") < 15_000).then(None).otherwise(pl.col("load_clean"))
).with_columns(load_filled=pl.col("load_clean").forward_fill(limit=4))

cleaned.select(
    raw_min=pl.col("load_mw").min(),
    raw_max=pl.col("load_mw").max(),
    clean_min=pl.col("load_clean").min(),
    clean_max=pl.col("load_clean").max(),
    masked=pl.col("load_clean").null_count() - pl.col("load_mw").null_count(),
    still_missing=pl.col("load_filled").null_count(),
)

### Lag features per region

In a long table every lag must stay inside its own group.

In [ ]:
(
    regional.sort("region", "timestamp")
    .select(
        "timestamp",
        "region",
        "load_mw",
        load_lag_1d=pl.col("load_mw").shift(48).over("region"),
        load_change_1d=pl.col("load_mw").diff(48).over("region"),
    )
    .filter(pl.col("timestamp") == datetime(2019, 1, 15, 18, 0, tzinfo=UTC))
)

### Try it 3.3: the same time one week earlier

Compare each Great Britain load value with the value **at the same time one week earlier**:
the earlier value, the difference in MW and the change in percent.

1. How many rows is one week on this grid?
2. On Christmas Day 2019 at 12:00 UTC, how much lower is the load than one week before?

<details>
<summary><b>Show the solution</b></summary>

```python
# One week is 336 rows only on a regular grid: check the time step before trusting shift().
print(gb["timestamp"].diff().drop_nulls().unique().to_list())

SAMPLES_PER_WEEK = 7 * 48

weekly_change = gb.select(
    "timestamp",
    "load_mw",
    week_earlier=pl.col("load_mw").shift(SAMPLES_PER_WEEK),
    change_mw=pl.col("load_mw").diff(SAMPLES_PER_WEEK),
    change_pct=(pl.col("load_mw").pct_change(SAMPLES_PER_WEEK) * 100).round(1),
)

weekly_change.filter(pl.col("timestamp") == datetime(2019, 12, 25, 12, 0, tzinfo=UTC))
```

One week is 7 x 48 = 336 rows. At noon on 25 December 2019 the load is 40,168 MW against
46,365 MW a week earlier: 6,197 MW less, or -13.4%.

`shift(336)` means "336 **rows** earlier". That equals one week only because the grid is
regular and complete, which is what the first line checks (a single time step, 30 minutes). On
a table with missing rows, join the table with itself on `timestamp + 7 days` instead (or
`upsample` first).

</details>

In [ ]:
# Your code here

# Part 4 - The lazy engine, speed and the rest

The lazy API and its optimiser, the streaming engine of Polars 2.0, a measured comparison with
pandas, and what they mean in practice.

## 4.1 The lazy API

Everything so far was **eager**: each call ran immediately on data in memory. In **lazy** mode
you describe the whole query on a `LazyFrame`; nothing runs until `collect()`. In between, the
query optimiser rewrites the plan. Two optimisations give most of the benefit:

- **projection pushdown**: read only the columns the query uses;
- **predicate pushdown**: apply filters while reading.

`pl.scan_csv` / `pl.scan_parquet` start a lazy query from a file; `df.lazy()` from a DataFrame.

In [ ]:
lazy_query = (
    pl.scan_csv(CSV_PATH, schema_overrides=schema_overrides, try_parse_dates=True)
    .filter(pl.col("utc_timestamp").dt.year() == 2019)
    .group_by(month=pl.col("utc_timestamp").dt.month())
    .agg(load_mean=pl.col("GB_GBN_load_actual_entsoe_transparency").mean())
    .sort("month")
)
print(type(lazy_query).__name__)     # nothing has been read yet

In [ ]:
# The optimised plan, read bottom-up: 2 of 41 columns are parsed (PROJECT) and the filter is
# applied inside the scan (SELECTION).
print(lazy_query.explain())

In [ ]:
# The output schema is known without touching the data: use it to validate a pipeline up front.
print(lazy_query.collect_schema())

lazy_query.collect().head(3)

### Convert once to Parquet

CSV carries no types and must be parsed every time. Parquet stores the schema, is compressed and
columnar, and lets the reader skip what a query does not need.

In [ ]:
PARQUET_PATH = DATA_DIR / "opsd_30min.parquet"
power.write_parquet(PARQUET_PATH)

print(f"CSV     {CSV_PATH.stat().st_size / 1e6:6.1f} MB")
print(f"Parquet {PARQUET_PATH.stat().st_size / 1e6:6.1f} MB")

timed("read_csv with declared schema", lambda: pl.read_csv(CSV_PATH, schema_overrides=schema_overrides, try_parse_dates=True))
timed("read_parquet", lambda: pl.read_parquet(PARQUET_PATH))
timed(
    "scan_parquet, 2 columns, one year",
    lambda: pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("utc_timestamp").dt.year() == 2019)
    .select("utc_timestamp", "IE_sem_price_day_ahead")
    .collect(),
)

## 4.2 Polars 2.0: streaming engine by default, and row order

Since Polars 2.0, `collect()` on a `LazyFrame` runs on the **streaming engine**: data is
processed in batches, so a query can handle inputs larger than RAM, and some operations (sorts,
window functions) can spill to disk. Eager `DataFrame` methods still use the in-memory engine.

The practical consequence: the streaming engine **does not guarantee row order** for operations
that do not need it. The upgrade guide names `join`, `unpivot` and `group_by`. The join is the one
that changes behaviour for existing code: in Polars 1.x a left join returned the rows in the
order of the left table; a lazy left join in 2.0 may not.

In [ ]:
REGIONAL_PATH = DATA_DIR / "regional.parquet"
regional.write_parquet(REGIONAL_PATH)
left_order = regional.select("timestamp", "region")


def keeps_left_order(result: pl.DataFrame) -> bool:
    """True if the rows come back in the same order as the left table."""
    return result.select("timestamp", "region").equals(left_order)


N_RUNS = 8
lazy_join = pl.scan_parquet(REGIONAL_PATH).join(regions.lazy(), on="region", how="left")
lazy_join_ordered = pl.scan_parquet(REGIONAL_PATH).join(
    regions.lazy(), on="region", how="left", maintain_order="left"
)

results = {
    "lazy join, default": sum(keeps_left_order(lazy_join.collect()) for _ in range(N_RUNS)),
    "lazy join, maintain_order='left'": sum(
        keeps_left_order(lazy_join_ordered.collect()) for _ in range(N_RUNS)
    ),
    "eager DataFrame.join": sum(
        keeps_left_order(regional.join(regions, on="region", how="left")) for _ in range(N_RUNS)
    ),
}
for label, hits in results.items():
    print(f"{label:<34} left order kept in {hits} of {N_RUNS} runs")

The first number changes from run to run and is usually far below 8. For time series this
matters: `diff`, `shift` and rolling windows assume sorted rows. **If the order matters, say so**:
pass `maintain_order` to the join, or sort explicitly afterwards.

`group_by` is a different story: it never returned the groups in a defined order, eager or lazy,
in Polars 1.x as well.

In [ ]:
group_orders = {
    tuple(regional.group_by("region").agg(pl.len())["region"]) for _ in range(N_RUNS)
}
print(f"eager group_by: {len(group_orders)} different group orders in {N_RUNS} runs")
print(regional.group_by("region", maintain_order=True).agg(n=pl.len())["region"].to_list())

## 4.3 How fast, really?

Three measurements, from small to large. Timings depend on the CPU, on the number of cores and on
the data, so **run the cells on your own machine**; the saved output comes from a small 2-core
machine, which is also what a free Colab runtime gives you.

### On this dataset: 400,000 rows

In [ ]:
def best_of(func, repeats: int = 3) -> float:
    """Return the best wall-clock time in seconds over `repeats` runs."""
    timings = []
    for _ in range(repeats):
        tic = time.perf_counter()
        func()
        timings.append(time.perf_counter() - tic)
    return min(timings)


def compare(benchmarks: dict, repeats: int = 3) -> None:
    """Time (pandas, polars) pairs of functions and print one line per pair."""
    print(f"{'operation':<38}{'pandas':>10}{'polars':>10}{'ratio':>8}")
    for label, (run_pandas, run_polars) in benchmarks.items():
        t_pandas, t_polars = best_of(run_pandas, repeats), best_of(run_polars, repeats)
        print(f"{label:<38}{t_pandas * 1000:>8.0f}ms{t_polars * 1000:>8.0f}ms{t_pandas / t_polars:>7.1f}x")


regional_sorted = regional.sort("region", "timestamp")
regional_pd = regional_sorted.to_pandas()
regional_pd_indexed = regional_pd.set_index("timestamp")

print(f"rows: {regional.height:,} | threads available to Polars: {pl.thread_pool_size()}\n")
compare(
    {
        "read the 37 MB CSV": (
            lambda: pd.read_csv(CSV_PATH),
            lambda: pl.read_csv(CSV_PATH, schema_overrides=schema_overrides, try_parse_dates=True),
        ),
        "mean by region, hour and weekday": (
            lambda: regional_pd.groupby(
                ["region", regional_pd["timestamp"].dt.hour, regional_pd["timestamp"].dt.dayofweek]
            )["load_mw"].mean(),
            lambda: regional_sorted.group_by(
                "region", hour=pl.col("timestamp").dt.hour(), weekday=pl.col("timestamp").dt.weekday()
            ).agg(pl.col("load_mw").mean()),
        ),
        "forward fill and diff, per region": (
            lambda: regional_pd.groupby("region")["load_mw"].ffill().groupby(regional_pd["region"]).diff(),
            lambda: regional_sorted.select(pl.col("load_mw").forward_fill().diff().over("region")),
        ),
        "hourly resample, per region": (
            lambda: regional_pd_indexed.groupby("region")["load_mw"].resample("1h").mean(),
            lambda: regional_sorted.group_by_dynamic("timestamp", every="1h", group_by="region").agg(
                pl.col("load_mw").mean()
            ),
        ),
    }
)

At this size the two libraries are in the same range: Polars is ahead on parsing the file and on
group operations, and pandas' `resample` holds its own. Nobody waits for either.

### At scale: 5 million rows

To see a difference we need more data. A synthetic measurement table: 1,000 sensors, one value per
minute, in arrival (shuffled) order, plus a small table of sensor metadata. Raise `N_STEPS` if your
machine has memory to spare.

In [ ]:
N_SENSORS = 1_000
N_STEPS = 5_000          # minutes: 1,000 x 5,000 = 5 million rows


def make_sensor_data(n_sensors: int, n_steps: int, seed: int = 0) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Synthetic long table (one row per sensor and minute, shuffled) and its metadata table."""
    rng = np.random.default_rng(seed)
    sensors = pl.DataFrame(
        {
            "sensor_id": [f"S{i:05d}" for i in range(n_sensors)],
            "site": [f"site_{i % 50:02d}" for i in range(n_sensors)],
            "kind": [("temperature", "pressure", "flow", "vibration")[i % 4] for i in range(n_sensors)],
        }
    )
    n_rows = n_sensors * n_steps
    times = pl.datetime_range(
        datetime(2026, 1, 1), datetime(2026, 1, 1) + timedelta(minutes=n_steps - 1), interval="1m", eager=True
    )
    measurements = (
        pl.DataFrame({"timestamp": times})
        .join(sensors.select("sensor_id"), how="cross")
        .with_columns(
            value=pl.Series(rng.normal(50, 10, n_rows)),
            quality=pl.Series(rng.integers(0, 100, n_rows, dtype=np.int32)),
        )
        .sample(fraction=1.0, shuffle=True, seed=seed)
    )
    return measurements, sensors


measurements, sensors = make_sensor_data(N_SENSORS, N_STEPS)
measurements_pd = measurements.to_pandas()
sensors_pd = sensors.to_pandas()

MEASUREMENTS_PATH = DATA_DIR / "measurements.parquet"
measurements.write_parquet(MEASUREMENTS_PATH)

print(f"rows: {measurements.height:,}")
print(f"in memory, Polars : {measurements.estimated_size('mb'):6.0f} MB")
print(f"in memory, pandas : {measurements_pd.memory_usage(deep=True).sum() / 1e6:6.0f} MB")
print(f"on disk, Parquet  : {MEASUREMENTS_PATH.stat().st_size / 1e6:6.0f} MB")
measurements.head(3)

The memory gap is the text column: Polars stores strings in Arrow buffers; pandas 2 stores one
Python object per value (pandas 3 has an Arrow-backed string type and is much closer).

Single operations first, each written the idiomatic way in both libraries.

In [ ]:
compare(
    {
        "filter two conditions, then arithmetic": (
            lambda: measurements_pd[
                (measurements_pd["quality"] > 10) & (measurements_pd["value"] > 55)
            ].assign(scaled=lambda d: d["value"] * 1.8 + 32),
            lambda: measurements.filter((pl.col("quality") > 10) & (pl.col("value") > 55)).with_columns(
                scaled=pl.col("value") * 1.8 + 32
            ),
        ),
        "text operation on 5 million strings": (
            lambda: measurements_pd["sensor_id"].str.slice(1).astype("int32"),
            lambda: measurements.select(pl.col("sensor_id").str.slice(1).cast(pl.Int32)),
        ),
        "join with the metadata table": (
            lambda: measurements_pd.merge(sensors_pd, on="sensor_id", how="left"),
            lambda: measurements.join(sensors, on="sensor_id", how="left"),
        ),
        "join, then mean by site and kind": (
            lambda: measurements_pd.merge(sensors_pd, on="sensor_id", how="left")
            .groupby(["site", "kind"])["value"]
            .mean(),
            lambda: measurements.join(sensors, on="sensor_id", how="left")
            .group_by("site", "kind")
            .agg(pl.col("value").mean()),
        ),
        "mean, std, max by sensor": (
            lambda: measurements_pd.groupby("sensor_id")["value"].agg(["mean", "std", "max"]),
            lambda: measurements.group_by("sensor_id").agg(
                mean=pl.col("value").mean(), std=pl.col("value").std(), max=pl.col("value").max()
            ),
        ),
        "sort by sensor and time": (
            lambda: measurements_pd.sort_values(["sensor_id", "timestamp"]),
            lambda: measurements.sort("sensor_id", "timestamp"),
        ),
    },
    repeats=2,
)

The picture is uneven, and worth reading as it is:

- **joins, text operations and filters** are several times faster in Polars, even on two cores;
- a **plain group-by** is roughly on par at this size and core count;
- the **two-key sort** is faster in pandas here.

So "Polars is always N times faster" is not a claim to repeat. What these single operations do
not show is the part that matters most in practice: a real analysis is a *chain* of them.

### A whole pipeline: the optimiser at work

The question: hourly mean per site, for temperature sensors only, keeping readings with
`quality >= 50`, starting from the Parquet file. Four ways to write it.

In [ ]:
def pandas_step_by_step() -> pd.DataFrame:
    """The natural way to write it: read, join, filter, group."""
    table = pd.read_parquet(MEASUREMENTS_PATH)
    return (
        table.merge(sensors_pd, on="sensor_id")
        .query("kind == 'temperature' and quality >= 50")
        .assign(hour=lambda d: d["timestamp"].dt.floor("h"))
        .groupby(["site", "hour"], as_index=False)["value"]
        .mean()
    )


def pandas_optimised_by_hand() -> pd.DataFrame:
    """The same result after reordering the steps by hand: filter early, join less."""
    table = pd.read_parquet(MEASUREMENTS_PATH, filters=[("quality", ">=", 50)])
    wanted = sensors_pd.loc[sensors_pd["kind"] == "temperature", ["sensor_id", "site"]]
    return (
        table.merge(wanted, on="sensor_id")
        .assign(hour=lambda d: d["timestamp"].dt.floor("h"))
        .groupby(["site", "hour"], as_index=False)["value"]
        .mean()
    )


def polars_eager() -> pl.DataFrame:
    """Polars, same order of steps as the first pandas version."""
    return (
        pl.read_parquet(MEASUREMENTS_PATH)
        .join(sensors, on="sensor_id")
        .filter((pl.col("kind") == "temperature") & (pl.col("quality") >= 50))
        .group_by("site", hour=pl.col("timestamp").dt.truncate("1h"))
        .agg(pl.col("value").mean())
    )


def polars_lazy_query() -> pl.LazyFrame:
    """The same steps in the same order, but lazy: only scan_parquet differs."""
    return (
        pl.scan_parquet(MEASUREMENTS_PATH)
        .join(sensors.lazy(), on="sensor_id")
        .filter((pl.col("kind") == "temperature") & (pl.col("quality") >= 50))
        .group_by("site", hour=pl.col("timestamp").dt.truncate("1h"))
        .agg(pl.col("value").mean())
    )


pipelines = {
    "pandas, step by step": pandas_step_by_step,
    "pandas, optimised by hand": pandas_optimised_by_hand,
    "polars, eager": polars_eager,
    "polars, lazy": lambda: polars_lazy_query().collect(),
}

reference = pandas_step_by_step().sort_values(["site", "hour"])["value"].to_numpy()
baseline = None
for label, run in pipelines.items():
    seconds = best_of(run)
    baseline = baseline or seconds
    result = run()
    values = (
        result.sort_values(["site", "hour"])["value"].to_numpy()
        if isinstance(result, pd.DataFrame)
        else result.sort("site", "hour")["value"].to_numpy()
    )
    same = np.allclose(values, reference)
    print(f"{label:<28}{seconds * 1000:>8.0f} ms   {baseline / seconds:>5.1f}x   same result: {same}")

In [ ]:
# Why the lazy version wins: read the plan bottom-up. The quality filter moved into the Parquet
# scan and the "temperature" filter moved onto the small table, so the join sees far fewer rows.
print(polars_lazy_query().explain())

The lazy query is the same code as the eager one, in the same order; the optimiser did what
`pandas_optimised_by_hand` does manually, and then ran it on all cores. That is the honest
summary of the speed difference:

- on **single operations** the gain ranges from nothing to about ten times, depending on the
  operation;
- on **pipelines read from files**, the lazy engine removes work you would otherwise have to
  remove by hand, and the gain is larger and comes for free;
- both grow with **more cores** (pandas uses one) and **more data**.

In [ ]:
del measurements_pd, regional_pd, regional_pd_indexed    # free memory before the next sections

## 4.4 Pitfalls of the first week

1. **Looking for the index.** There is none. Keys are ordinary columns; alignment is done with
   explicit joins, time-series tools take the name of the time column.
2. **Assigning in place.** `df["new"] = ...` raises. Use `with_columns` and rebind the name.
3. **Using a column created in the same context.** Expressions in one context run in parallel:
   use a second `with_columns`.
4. **Relying on row order after `group_by` / `join`.** Sort, or pass `maintain_order`
   (`group_by` never had a defined order; lazy joins lost theirs in Polars 2.0).
5. **Treating `NaN` as missing.** Convert to `null` on the way in.
6. **Guessed schemas and guessed date formats.** Declare both.
7. **Bare strings in `then()`.** They are column names. Use `pl.lit("text")`.
8. **Forgetting parentheses** around comparisons combined with `&` / `|`.
9. **Python functions row by row** (`map_elements`). It works, but it leaves the fast path.
   Polars itself warns about it and, for simple lambdas, prints the native rewrite:

In [ ]:
sample = gb.select("load_mw").drop_nulls()


def with_python_lambda() -> pl.DataFrame:
    return sample.select(pl.col("load_mw").map_elements(lambda mw: mw / 1000, return_dtype=pl.Float64))


def with_native_expression() -> pl.DataFrame:
    return sample.select(pl.col("load_mw") / 1000)


t_lambda = best_of(with_python_lambda)
t_native = best_of(with_native_expression)
print(f"map_elements      {t_lambda * 1000:8.2f} ms")
print(f"native expression {t_native * 1000:8.2f} ms   ({t_lambda / t_native:.0f}x faster)")

In [ ]:
# In-place assignment is not supported: the error message points to the right method.
try:
    gb["load_gw"] = gb["load_mw"] / 1000
except TypeError as err:
    print("TypeError:", " ".join(str(err).split()))

If a computation truly has no expression equivalent, `map_batches` passes a whole `Series` to
your function (for example a NumPy or SciPy routine), which keeps the work vectorised.

## 4.5 Interop: NumPy, pandas, SQL

Conversion to NumPy and pandas is one method call, which is all scikit-learn, PyTorch and
matplotlib need. `to_pandas()` and `from_pandas()` go through Arrow and need `pyarrow`.

In [ ]:
features = gb.drop_nulls().select(cs.float())

X = features.to_numpy()                       # 2-D array for scikit-learn / PyTorch
print(type(X).__name__, X.shape, X.dtype)

as_pandas = gb.head(3).to_pandas()
back_to_polars = pl.from_pandas(as_pandas)
print(type(as_pandas).__name__, "->", type(back_to_polars).__name__)

In [ ]:
# SQL runs on the same engine as the expression API.
pl.sql(
    """
    SELECT region,
           ROUND(AVG(load_mw), 1) AS load_mean,
           COUNT(load_mw)         AS n_valid
    FROM regional
    GROUP BY region
    ORDER BY load_mean DESC
    """
).collect()

## 4.6 The same query in pandas, Polars and PySpark

Mean load per region in 2019, highest first. PySpark is outside the scope of the session; it is
here for orientation, because its syntax is the closest relative of Polars: column expressions,
lazy by construction, no index.

In [ ]:
regional_pandas = regional.to_pandas()

by_region_pandas = (
    regional_pandas[regional_pandas["timestamp"].dt.year == 2019]
    .groupby("region", as_index=False)
    .agg(load_mean=("load_mw", "mean"))
    .sort_values("load_mean", ascending=False)
)

by_region_polars = (
    regional.filter(pl.col("timestamp").dt.year() == 2019)
    .group_by("region")
    .agg(load_mean=pl.col("load_mw").mean())
    .sort("load_mean", descending=True)
)

print(by_region_pandas.round(1).to_string(index=False))
by_region_polars

The PySpark version is not executed here (it needs a JVM and a Spark session). It was run
separately on Spark 4.2 against `data/regional.parquet` and returns the same four numbers.

```python
from pyspark.sql import functions as F

by_region_spark = (
    spark.read.parquet("data/regional.parquet")
    .filter(F.year("timestamp") == 2019)
    .groupBy("region")
    .agg(F.avg("load_mw").alias("load_mean"))
    .orderBy(F.desc("load_mean"))
)
```

`F.year` uses the Spark session time zone: set `spark.sql.session.timeZone` to `UTC` to match.

## 4.7 Putting it together: one query, step by step

The question: **in each region, which three days of 2019 had the highest peak load compared
with the usual peak for that weekday?**

It uses almost everything seen so far: a lazy scan, a sanity check per group, resampling per
region, a group statistic written on every row, a rank inside each group. In words first:

1. point at the Parquet file, lazily;
2. keep the rows of 2019;
3. turn implausible samples into null: those below 30% of the median of their own region;
4. sort by region and time;
5. make one row per region and day, with the daily peak and the number of valid samples;
6. keep complete days only: 48 valid half-hours;
7. add the usual peak: the median per region and weekday, on every row;
8. add the excess: peak over usual, minus 1;
9. keep the three highest days of each region;
10. put the result in order, then run the query.

Three small expressions come first, as building blocks; the chain then reads almost like the
list above.

In [ ]:
load = pl.col("load_mw")
plausible = load > 0.3 * load.median().over("region")     # above 30% of the median of its own region
weekday = pl.col("timestamp").dt.weekday()                # Monday = 1 ... Sunday = 7

top_days_query = (
    # 1. Lazy scan: nothing is read yet, the query is only described.
    pl.scan_parquet(REGIONAL_PATH)
    # 2. Rows of 2019. The optimiser applies this filter inside the scan.
    .filter(pl.col("timestamp").dt.year() == 2019)
    # 3. Implausible samples become null: when/then without otherwise gives null.
    #    The threshold is per region, because the regions differ in size by a factor of about 70.
    .with_columns(load_mw=pl.when(plausible).then(load))
    # 4. group_by_dynamic needs every group sorted by time.
    .sort("region", "timestamp")
    # 5. Resample: one row per region and day, with the daily peak and the number of valid samples.
    .group_by_dynamic("timestamp", every="1d", group_by="region")
    .agg(peak=load.max(), n_valid=load.count())
    # 6. Complete days only: 48 valid half-hours.
    .filter(pl.col("n_valid") == 48)
    # 7. The usual peak: the median per region and weekday, written on every row by .over().
    .with_columns(usual=pl.col("peak").median().over("region", weekday))
    # 8. The excess of each day over its usual peak. A second context: "usual" must exist first.
    .with_columns(excess=pl.col("peak") / pl.col("usual") - 1)
    # 9. A rank inside each region, used directly as a filter: the three highest days.
    .filter(pl.col("excess").rank(descending=True).over("region") <= 3)
    # 10. An explicit order: a lazy query does not promise one.
    .sort("region", "excess", descending=[False, True])
)

top_days = top_days_query.collect()                       # now it runs
with pl.Config(tbl_rows=12):                              # show all twelve rows
    print(top_days.with_columns(pl.col("excess").round(3)))

Twelve rows, three per region, from 403,208. What the query found:

- **Cyprus**: 27, 29 and 30 August, about 37% above the usual peak. Demand there peaks in the
  summer heat.
- **Great Britain**: 20 January and 3 February, two Sundays 25% above a usual Sunday, and
  31 January. The 51,467 MW of 31 January is the highest value of 2019, the one Try it 1.2 found.
- **Northern Ireland**: 17 and 19 January and 2 February, about 32% above.
- **Ireland**: 13 and 16 November and 17 December, 22 to 23% above.

The plan shows what the engine did with the description:

In [ ]:
print(top_days_query.explain())

Read it from the bottom. The scan reads **3 of the 4 columns** (`wind_onshore_mw` is never used)
and applies the filter on the year while reading (`SELECTION`): both were decided by the
optimiser, not written in the query.

One honest limit of this query: the usual peak is a median over the whole year, so what it finds
is mostly the season, winter in the north and late summer in Cyprus. To compare each day with the
same weekday *of its own month*, one expression changes: in step 7 write
`.over("region", weekday, pl.col("timestamp").dt.month())`. Try it and see which days remain.

# 5. Coming from pandas: a cheat sheet

The sections above are written Polars first. This table is the map for those who know the pandas
names.

| Topic | pandas | Polars |
|-------|--------|--------|
| Inspect | `head`, `tail`, `shape`, `describe` | same names |
| | `dtypes`, `info()` | `schema`, `glimpse()`, `null_count()` |
| Dates | `pd.to_datetime(s, format=...)` | `pl.col("s").str.to_datetime(format)` |
| | `dayfirst=True` | explicit `format` |
| | `errors="coerce"` | `strict=False` |
| | `.dt.hour`, `.dt.dayofweek` (Mon = 0) | `.dt.hour()`, `.dt.weekday()` (Mon = 1) |
| | `.dt.tz_convert(tz)` | `.dt.convert_time_zone(tz)` |
| Cleaning | `isna`, `dropna`, `fillna` | `is_null`, `drop_nulls`, `fill_null` |
| | `ffill`, `bfill` | `forward_fill`, `backward_fill` |
| | `interpolate(method="time")` | `interpolate_by("timestamp")` |
| | `duplicated`, `drop_duplicates` | `is_duplicated`, `unique` |
| | `sort_index()` | `sort("timestamp")` |
| | `astype`, `pd.to_numeric(errors="coerce")` | `cast`, `cast(..., strict=False)` |
| Columns | `df["c"] = a * b`, `assign` | `with_columns(c=pl.col("a") * pl.col("b"))` |
| | index alignment | none: `join` explicitly |
| | `rename(columns=...)`, `drop(columns=...)` | `rename(...)`, `drop(...)` |
| Statistics | `sum`, `mean`, `min`, `max`, `std`, `quantile` | same names, as expressions |
| Plot | `df.plot()` | matplotlib, or plotly express: `px.line(df, x=..., y=...)` |
| Combine | `pd.concat`, `merge`, `merge_asof` | `pl.concat`, `join`, `join_asof` |
| Group | `groupby().agg()`, `transform` | `group_by().agg()`, `.over()` |
| Reshape | `pivot`, `pivot_table`, `melt` | `pivot`, `unpivot` |
| Resample | `resample("1h").mean()` | `group_by_dynamic("timestamp", every="1h").agg(...)` |
| | `asfreq` | `upsample` |
| | `pd.date_range` | `pl.datetime_range`, `pl.date_range` |
| | `.ohlc()` | `first`, `max`, `min`, `last` in `agg` |
| Windows | `rolling(n)`, `rolling("10min")` | `rolling_mean(n)`, `rolling_mean_by("timestamp", "10m")` |
| | `min_periods`, `center` | `min_samples`, `center` |
| | `ewm(span=...).mean()` | `ewm_mean(span=...)` |
| | `expanding().max()` | `cum_max()` |
| Lags | `shift`, `diff`, `pct_change` | same names |
| No pandas counterpart | | `scan_*` / `collect` (lazy), `explain()`, `sink_parquet` |

# 6. Exercises

Three longer exercises that combine several sections. As everywhere in this notebook, each
solution is folded: click *Show the solution* only after trying.

### Exercise 1: grouping

For each region, find the hour of the day (UTC) with the highest average load, and that average.

<details>
<summary><b>Show the solution</b></summary>

```python
(
    regional
    .group_by("region", hour=pl.col("timestamp").dt.hour())
    .agg(load_mean=pl.col("load_mw").mean().round(1))
    .sort("load_mean", descending=True)
    .group_by("region", maintain_order=True)
    .first()
)
```

The peak hour is 17:00 UTC for Great Britain (41,863 MW), Ireland (3,778 MW) and Cyprus
(609 MW), and 18:00 UTC for Northern Ireland (1,146 MW).

Two steps: the mean per region and hour, then "the best row of each region". The second step
sorts and takes the first row of every group; `maintain_order=True` is what makes "first"
mean "first in the sorted order".

</details>

In [ ]:
# Your code here

### Exercise 2: resampling and windows

For Great Britain, compute the daily energy in GWh (each half-hour at `P` MW is `P / 2` MWh) and
its 7-day rolling mean. Which are the three days with the highest daily energy?

<details>
<summary><b>Show the solution</b></summary>

```python
daily_energy = (
    gb.sort("timestamp")
    .group_by_dynamic("timestamp", every="1d")
    .agg(energy_gwh=(pl.col("load_mw") * 0.5).sum() / 1000, n_valid=pl.col("load_mw").count())
    .with_columns(energy_7d=pl.col("energy_gwh").rolling_mean_by("timestamp", window_size="7d"))
)
daily_energy.top_k(3, by="energy_gwh").with_columns(cs.float().round(1))
```

5 February 2015 (1,121.5 GWh), 2 February 2015 (1,118.1 GWh) and 20 January 2015
(1,116.6 GWh), all with 48 valid samples.

Keep `n_valid`: a day with missing half-hours has a lower *sum* for no physical reason. A sum
is sensitive to gaps in a way a mean is not.

</details>

In [ ]:
# Your code here

### Exercise 3: lazy

Starting from `pl.scan_parquet(PARQUET_PATH)`, compute the monthly mean of the day-ahead price
`IE_sem_price_day_ahead` for 2017. Print the optimised plan and check how many columns are read.

<details>
<summary><b>Show the solution</b></summary>

```python
price_2017 = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("utc_timestamp").dt.year() == 2017)
    .group_by(month=pl.col("utc_timestamp").dt.month())
    .agg(price_mean=pl.col("IE_sem_price_day_ahead").mean().round(2))
    .sort("month")
)
print(price_2017.explain())
price_2017.collect().head(4)
```

The plan reads `PROJECT 2/41 COLUMNS` and carries the year filter as a `SELECTION` inside the
Parquet scan: two columns out of 41 are read, and the filter is applied while reading. The
first four months are 53.95, 51.51, 44.25 and 43.85 EUR/MWh.

</details>

In [ ]:
# Your code here

## Where to go next

- User guide: https://docs.pola.rs/user-guide/
- Coming from pandas: https://docs.pola.rs/user-guide/migration/pandas/
- Python API reference: https://docs.pola.rs/api/python/stable/reference/
- Polars 2.0 upgrade guide: https://docs.pola.rs/releases/upgrade/2/
- The dataset: https://data.open-power-system-data.org/time_series/2020-10-06